# BP 因子

BP因子：净资产/总市值，该因子将会和市值高度相关，于是在检测该因子时可以很自然的给出做市值中性化的理由，如若不然，该因子很大程度上就是在计算市值因子，所以首先考虑将该因子进行市值中性化后去计算因子的各项指标

## 市值中性化后因子IC均值、ICIR、RankIC均值、RankICIR、因子收益率、t值

In [ ]:
# -*- coding: utf-8 -*-
"""
BigQuant BP因子复现（N日频版本，无文件保存输出）

修改点：
- 不再使用自然月截面，因此删除 next_month 逻辑。
- 使用 N_FREQ=30 个交易日频率：
  1. 在第T个截面日计算BP暴露；
  2. 用未来30个交易日收益作为因变量；
  3. 每30个交易日形成一个截面。

流程：
BP = 1/PB
-> 市值中性化 BP ~ log(MarketCap)
-> 中位数MAD去极值
-> Z-score标准化
-> 缺失值填0
-> IC / RankIC / 因子收益率 / t值
-> 仅输出汇总指标表
-> IC与RankIC时序图
"""

import warnings
from typing import Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

try:
    from IPython.display import display, Markdown
    IPYTHON = True
except Exception:
    IPYTHON = False

try:
    import dai
except Exception:
    dai = None

warnings.filterwarnings("ignore")

# ================= 参数 =================
START_DATE = "2022-01-01"
END_DATE = "2026-06-30"
N_FREQ = 60                      # N日频核心参数
USE_EXCESS_RETURN = True
BENCHMARK = "000300.SH"
WINSOR_K = 5
MIN_CS_COUNT = 30


# ================= 展示 =================
def show_table(df, title):
    if IPYTHON:
        display(Markdown("### " + title))
        display(df.style.format(precision=6))
    else:
        print(title)
        print(df)


def set_chinese_font():
    fonts = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "WenQuanYi Micro Hei"]
    installed = {x.name for x in font_manager.fontManager.ttflist}
    for f in fonts:
        if f in installed:
            plt.rcParams["font.sans-serif"] = [f]
            break
    plt.rcParams["axes.unicode_minus"] = False


# ================= 数据读取 =================
def query_stock():
    sql = f"""
    SELECT
        v.date,
        v.instrument,
        v.pb,
        v.total_market_cap,
        v.float_market_cap,
        b.close
    FROM cn_stock_valuation v
    JOIN cn_stock_bar1d b
      ON v.date=b.date AND v.instrument=b.instrument
    WHERE v.date BETWEEN '{START_DATE}' AND '{END_DATE}'
    """
    return dai.query(sql).df()


def query_index():
    sql = f"""
    SELECT date, close
    FROM cn_stock_index_bar1d
    WHERE instrument='{BENCHMARK}'
      AND date BETWEEN '{START_DATE}' AND '{END_DATE}'
    """
    idx = dai.query(sql).df()
    idx["date"] = pd.to_datetime(idx["date"])
    idx = idx.sort_values("date")
    idx["benchmark_ret"] = idx["close"].shift(-N_FREQ) / idx["close"] - 1
    return idx[["date", "benchmark_ret"]]


# ================= 因子处理 =================
def neutralize(y, x):
    mask = y.notna() & x.notna()
    out = pd.Series(np.nan, index=y.index)
    if mask.sum() < MIN_CS_COUNT:
        return y - y.mean()
    X = np.column_stack([np.ones(mask.sum()), x[mask]])
    beta = np.linalg.lstsq(X, y[mask], rcond=None)[0]
    out.loc[mask] = y[mask] - X @ beta
    return out


def winsor(x):
    med = x.median()
    mad = (x-med).abs().median()
    if mad == 0 or np.isnan(mad):
        return x
    return x.clip(med-WINSOR_K*mad, med+WINSOR_K*mad)


def zscore(x):
    return (x-x.mean()) / x.std(ddof=0)


def process_cross_section(df):
    df=df.copy()
    df["bp_size_neutral"] = neutralize(
        df["bp_raw"],
        np.log(df["total_market_cap"])
    )
    df["bp_final"] = zscore(winsor(df["bp_size_neutral"]))
    df["bp_final"] = df["bp_final"].replace([np.inf,-np.inf],np.nan).fillna(0)
    return df


# ================= 构造N日频截面 =================
def build_nday_panel(raw):
    raw=raw.copy()
    raw["date"] = pd.to_datetime(raw["date"])
    raw=raw.sort_values(["instrument","date"])

    raw["bp_raw"] = np.where(raw["pb"]>0,1/raw["pb"],np.nan)

    # 每只股票未来N日收益
    raw["future_close"] = raw.groupby("instrument")["close"].shift(-N_FREQ)
    raw["fwd_ret"] = raw["future_close"]/raw["close"]-1

    # 全市场交易日序号，用于形成不重叠N日截面
    trading_days = pd.DataFrame({
        "date": sorted(raw["date"].unique())
    })
    trading_days["idx"] = np.arange(len(trading_days))
    trading_days["use"] = trading_days["idx"] % N_FREQ == 0

    raw = raw.merge(
        trading_days[trading_days["use"]][["date"]],
        on="date",
        how="inner"
    )

    return raw


# ================= 回归和统计 =================
def wls(y,x,w):
    mask=y.notna() & x.notna() & w.notna()
    if mask.sum()<MIN_CS_COUNT:
        return np.nan,np.nan
    X=np.column_stack([np.ones(mask.sum()),x[mask]])
    Y=y[mask].values
    W=np.sqrt(w[mask].values)
    Xw=X*W[:,None]
    Yw=Y*W
    beta=np.linalg.lstsq(Xw,Yw,rcond=None)[0]
    pred=X@beta
    resid=Y-pred
    sigma=np.sqrt((resid**2).sum()/(mask.sum()-2))
    cov=np.linalg.inv(Xw.T@Xw)*sigma**2
    se=np.sqrt(np.diag(cov))
    return beta[1], beta[1]/se[1]


def run_test(panel):
    results=[]
    processed=[]

    for d,g in panel.groupby("date"):
        g=process_cross_section(g)
        processed.append(g)
        valid=g[["bp_final","fwd_ret"]].dropna()

        ic=valid["bp_final"].corr(valid["fwd_ret"])
        ric=valid["bp_final"].corr(valid["fwd_ret"],method="spearman")

        fr,t=wls(
            g["fwd_ret"],
            g["bp_final"],
            np.sqrt(g["float_market_cap"])
        )

        results.append({
            "截面日期":d,
            "IC":ic,
            "RankIC":ric,
            "因子收益率":fr,
            "t值":t,
            "样本数":len(valid)
        })

    ts=pd.DataFrame(results)

    summary=pd.DataFrame([{
        "IC均值":ts.IC.mean(),
        "ICIR":ts.IC.mean()/ts.IC.std(),
        "RankIC均值":ts.RankIC.mean(),
        "RankICIR":ts.RankIC.mean()/ts.RankIC.std(),
        "因子收益率均值":ts["因子收益率"].mean(),
        "因子收益率t值":ts["因子收益率"].mean()/(ts["因子收益率"].std()/np.sqrt(len(ts)))
    }])

    return summary,ts,pd.concat(processed)


# ================= 主函数 =================
def main():
    raw=query_stock()
    panel=build_nday_panel(raw)

    if USE_EXCESS_RETURN:
        idx=query_index()
        panel=panel.merge(idx,on="date",how="left")
        panel["target_ret"]=panel["fwd_ret"]-panel["benchmark_ret"]
        panel["fwd_ret"]=panel["target_ret"]

    summary,ts,processed=run_test(panel)

    show_table(summary,"BP因子N日频汇总指标")

    set_chinese_font()
    plt.figure(figsize=(14,5))
    plt.plot(ts["截面日期"],ts["IC"],label="IC")
    plt.plot(ts["截面日期"],ts["RankIC"],label="RankIC")
    plt.axhline(0,linestyle="--")
    plt.title(f"BP因子 {N_FREQ}日频 IC 与 RankIC")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    # 不主动保存任何结果文件，仅返回 DataFrame 和图表。
    # 如需保存，可在外部自行调用 DataFrame.to_csv()。
    return summary,ts,processed


summary_table,timeseries_table,processed_factor_panel=main()


结合图中结果来看，该因子以60日为截面周期时表现较好，因子IC与RankIC主要为正，RankIC均值表现良好，可以初步认为该因子是一个相对较好的低频估值类因子，但接下来为了进一步做好因子工程，可以考虑再做一次行业中性化

## 市值行业中性化后因子各指标计算

In [ ]:
# -*- coding: utf-8 -*-
"""
BigQuant BP因子复现（N日频版本，市值+行业中性化 v7修正版）

修改点：
- 不再使用自然月截面，因此删除 next_month 逻辑。
- 使用 N_FREQ=30 个交易日频率：
  1. 在第T个截面日计算BP暴露；
  2. 用未来30个交易日收益作为因变量；
  3. 每30个交易日形成一个截面。

流程：
BP = 1/PB
-> 市值+行业中性化 BP ~ log(MarketCap)+Industry
-> 中位数MAD去极值
-> Z-score标准化
-> 缺失值填0
-> IC / RankIC / 因子收益率 / t值
-> 仅输出汇总指标表
-> IC与RankIC时序图
"""

import warnings
from typing import Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

try:
    from IPython.display import display, Markdown
    IPYTHON = True
except Exception:
    IPYTHON = False

try:
    import dai
except Exception:
    dai = None

warnings.filterwarnings("ignore")

# ================= 参数 =================
START_DATE = "2020-07-01"
END_DATE = "2026-06-30"
N_FREQ = 60                      # N日频核心参数
USE_EXCESS_RETURN = True
BENCHMARK = "000300.SH"
WINSOR_K = 5
MIN_CS_COUNT = 30


# ================= 展示 =================
def show_table(df, title):
    if IPYTHON:
        display(Markdown("### " + title))
        display(df.style.format(precision=6))
    else:
        print(title)
        print(df)


def set_chinese_font():
    fonts = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "WenQuanYi Micro Hei"]
    installed = {x.name for x in font_manager.fontManager.ttflist}
    for f in fonts:
        if f in installed:
            plt.rcParams["font.sans-serif"] = [f]
            break
    plt.rcParams["axes.unicode_minus"] = False


# ================= 数据读取 =================
def query_stock():
    sql = f"""
    SELECT
        v.date,
        v.instrument,
        v.pb,
        v.total_market_cap,
        v.float_market_cap,
        b.close,
        p.cs_level1 AS industry
    FROM cn_stock_valuation v
    JOIN cn_stock_bar1d b
      ON v.date=b.date AND v.instrument=b.instrument
    LEFT JOIN cn_stock_prefactors p
      ON v.date=p.date AND v.instrument=p.instrument
    WHERE v.date BETWEEN '{START_DATE}' AND '{END_DATE}'
    """
    return dai.query(sql).df()


def query_index():
    sql = f"""
    SELECT date, close
    FROM cn_stock_index_bar1d
    WHERE instrument='{BENCHMARK}'
      AND date BETWEEN '{START_DATE}' AND '{END_DATE}'
    """
    idx = dai.query(sql).df()
    idx["date"] = pd.to_datetime(idx["date"])
    idx = idx.sort_values("date")
    idx["benchmark_ret"] = idx["close"].shift(-N_FREQ) / idx["close"] - 1
    return idx[["date", "benchmark_ret"]]


# ================= 因子处理 =================
def neutralize_size_industry(df):
    """
    市值+行业中性化：
    BP_raw ~ log(total_market_cap) + 行业哑变量
    返回残差作为纯BP暴露。
    """
    y = df["bp_raw"]
    log_mv = np.log(df["total_market_cap"])
    industry = df["industry"].fillna("unknown").astype(str)

    mask = y.notna() & log_mv.notna()
    out = pd.Series(np.nan, index=df.index)

    if mask.sum() < MIN_CS_COUNT:
        return y - y.mean()

    X_ind = pd.get_dummies(industry[mask], drop_first=True).astype(float)
    X = pd.concat([
        pd.Series(1.0, index=df.index[mask], name="const"),
        log_mv[mask].rename("log_mv"),
        X_ind
    ], axis=1)

    beta = np.linalg.lstsq(
        X.values,
        y[mask].values,
        rcond=None
    )[0]

    out.loc[mask] = y[mask].values - X.values @ beta
    return out


def winsor(x):
    med = x.median()
    mad = (x-med).abs().median()
    if mad == 0 or np.isnan(mad):
        return x
    return x.clip(med-WINSOR_K*mad, med+WINSOR_K*mad)


def zscore(x):
    return (x-x.mean()) / x.std(ddof=0)


def process_cross_section(df):
    df=df.copy()
    # 市值+行业中性化
    df["bp_size_industry_neutral"] = neutralize_size_industry(df)

    # 去极值 + 标准化 + 缺失值填0
    df["bp_final"] = zscore(winsor(df["bp_size_industry_neutral"]))
    df["bp_final"] = df["bp_final"].replace([np.inf,-np.inf],np.nan).fillna(0)
    return df


# ================= 构造N日频截面 =================
def build_nday_panel(raw):
    raw=raw.copy()
    raw["date"] = pd.to_datetime(raw["date"])
    raw=raw.sort_values(["instrument","date"])

    raw["bp_raw"] = np.where(raw["pb"]>0,1/raw["pb"],np.nan)

    # 每只股票未来N日收益
    raw["future_close"] = raw.groupby("instrument")["close"].shift(-N_FREQ)
    raw["fwd_ret"] = raw["future_close"]/raw["close"]-1

    # 全市场交易日序号，用于形成不重叠N日截面
    trading_days = pd.DataFrame({
        "date": sorted(raw["date"].unique())
    })
    trading_days["idx"] = np.arange(len(trading_days))
    trading_days["use"] = trading_days["idx"] % N_FREQ == 0

    raw = raw.merge(
        trading_days[trading_days["use"]][["date"]],
        on="date",
        how="inner"
    )

    return raw


# ================= 回归和统计 =================
def wls(y,x,w):
    mask=y.notna() & x.notna() & w.notna()
    if mask.sum()<MIN_CS_COUNT:
        return np.nan,np.nan
    X=np.column_stack([np.ones(mask.sum()),x[mask]])
    Y=y[mask].values
    W=np.sqrt(w[mask].values)
    Xw=X*W[:,None]
    Yw=Y*W
    beta=np.linalg.lstsq(Xw,Yw,rcond=None)[0]
    pred=X@beta
    resid=Y-pred
    sigma=np.sqrt((resid**2).sum()/(mask.sum()-2))
    cov=np.linalg.inv(Xw.T@Xw)*sigma**2
    se=np.sqrt(np.diag(cov))
    return beta[1], beta[1]/se[1]


def run_test(panel):
    results=[]
    processed=[]

    for d,g in panel.groupby("date"):
        g=process_cross_section(g)
        processed.append(g)
        valid=g[["bp_final","fwd_ret"]].dropna()

        ic=valid["bp_final"].corr(valid["fwd_ret"])
        ric=valid["bp_final"].corr(valid["fwd_ret"],method="spearman")

        fr,t=wls(
            g["fwd_ret"],
            g["bp_final"],
            np.sqrt(g["float_market_cap"])
        )

        results.append({
            "截面日期":d,
            "IC":ic,
            "RankIC":ric,
            "因子收益率":fr,
            "t值":t,
            "样本数":len(valid)
        })

    ts=pd.DataFrame(results)

    summary=pd.DataFrame([{
        "IC均值":ts.IC.mean(),
        "ICIR":ts.IC.mean()/ts.IC.std(),
        "RankIC均值":ts.RankIC.mean(),
        "RankICIR":ts.RankIC.mean()/ts.RankIC.std(),
        "因子收益率均值":ts["因子收益率"].mean(),
        "因子收益率t值":ts["因子收益率"].mean()/(ts["因子收益率"].std()/np.sqrt(len(ts)))
    }])

    return summary,ts,pd.concat(processed)


# ================= 主函数 =================
def main():
    raw=query_stock()
    panel=build_nday_panel(raw)

    if USE_EXCESS_RETURN:
        idx=query_index()
        panel=panel.merge(idx,on="date",how="left")
        panel["target_ret"]=panel["fwd_ret"]-panel["benchmark_ret"]
        panel["fwd_ret"]=panel["target_ret"]

    summary,ts,processed=run_test(panel)

    show_table(summary,"BP因子N日频汇总指标")

    set_chinese_font()
    plt.figure(figsize=(14,5))
    plt.plot(ts["截面日期"],ts["IC"],label="IC")
    plt.plot(ts["截面日期"],ts["RankIC"],label="RankIC")
    plt.axhline(0,linestyle="--")
    plt.title(f"BP因子 {N_FREQ}日频 IC 与 RankIC")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    # 不主动保存任何结果文件，仅返回 DataFrame 和图表。
    # 如需保存，可在外部自行调用 DataFrame.to_csv()。
    return summary,ts,processed


summary_table,timeseries_table,processed_factor_panel=main()


虽然在行业中性化后因子的IC和RankIC均值都有一定的下降，但换来了ICIR和RankICIR的上升，这说明因子变得更加稳定，同时还能将因子变得更加纯净，为后续的多因子组合工作做好准备，于是后续将主要采用市值行业中性化后的因子来进行研究。

## 市值分层回测

In [ ]:
# -*- coding: utf-8 -*-
"""
BP 因子：市值行业中性化后按市值分层选股回测（性能优化版，不展示中间表，修复DAI filters）

相对上一版的优化点
----------------
1. 合并数据查询与 DAI 分区过滤：
   - 所有 DAI 查询统一传入 filters={"date": [START_DATE, END_DATE]}，避免 cn_stock_bar1d 分区权限错误；
   - 不再分别查询 PB/市值、行业、状态后多次 merge；
   - 默认一次 SQL 读取信号日所需字段：PB、市值、行业、证券简称。
2. 减少兼容性探测：
   - 默认走 BigQuant 常用字段 fast path；
   - 只在 fast path 失败时尝试少量备用 SQL。
3. 减少内存占用：
   - 只读取调仓信号日截面，不读取全量日频面板；
   - 只保留必要列；
   - 数值列 downcast 到 float32；
   - 行业列转 category。
4. 优化中性化：
   - 使用 FWL 方式剔除行业固定效应和市值暴露；
   - 避免为每个截面构造大规模行业 dummy 矩阵。
5. 减少日志和中间展示：
   - VERBOSE 控制进度输出；
   - 不主动保存任何结果文件。

策略逻辑
--------
- BP = 1 / PB；
- 信号日剔除可识别 ST、*ST、退市股票；
- 每个截面做：市值 + 行业中性化 -> MAD去极值 -> Z-score标准化；
- 全市场按总市值从小到大分为 15 组；
- 通过 SIZE_GROUPS_TO_TRADE 指定交易市值组；
- 每个指定市值组内，买入因子值最大的前 10%；
- 调仓周期 60 个交易日；
- 交易撮合、涨跌停、停牌由 BigTrader 原生回测引擎处理；
- 不读取下一交易日涨跌停、停牌、成交量等信息，避免未来函数。
"""

import warnings
warnings.filterwarnings("ignore")

import gc
import time
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

import dai
from bigquant import bigtrader


# =========================
# 1. 参数设置
# =========================

START_DATE = "2020-01-01"
END_DATE = "2026-06-30"

FACTOR_NAME = "bp_raw"
NEUTRAL_FACTOR_NAME = "bp_mkt_ind_neutral"

# 1=最小市值组，15=最大市值组
SIZE_GROUPS_TO_TRADE = [1,2]
N_SIZE_GROUPS = 15

TOP_PCT = 0.15
REBALANCE_DAYS = 60

SELECT_COUNT_METHOD = "floor"      # "floor" or "ceil"
GROUP_CAPITAL_EQUAL = True         # True：市值组等资金；False：全部入选股票整体等权

WINSOR_K = 5
MIN_OBS_PER_CROSS_SECTION = 100
MIN_STOCKS_PER_SIZE_GROUP = 20

CAPITAL_BASE = 1_000_000
BENCHMARK = "000300.SH"

BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COMMISSION = 5

# 性能开关
VERBOSE = True
DISPLAY_SIGNAL_SUMMARY = False          # False：不打印中间交易信号摘要表
FAST_QUERY_ONLY = False             # True：只走最快字段组合；False：fast path失败后尝试少量备用字段


START_DATE = pd.to_datetime(START_DATE).strftime("%Y-%m-%d")
END_DATE = pd.to_datetime(END_DATE).strftime("%Y-%m-%d")

if not SIZE_GROUPS_TO_TRADE:
    raise ValueError("SIZE_GROUPS_TO_TRADE 不能为空。")

bad_groups = [g for g in SIZE_GROUPS_TO_TRADE if int(g) < 1 or int(g) > N_SIZE_GROUPS]
if bad_groups:
    raise ValueError(f"SIZE_GROUPS_TO_TRADE 中存在非法市值组：{bad_groups}，有效范围为 1~{N_SIZE_GROUPS}。")

SIZE_GROUPS_TO_TRADE = sorted(set([int(x) for x in SIZE_GROUPS_TO_TRADE]))

if SELECT_COUNT_METHOD not in ["floor", "ceil"]:
    raise ValueError("SELECT_COUNT_METHOD 只能是 'floor' 或 'ceil'。")


# =========================
# 2. 工具函数
# =========================

_T0 = time.time()


def _elapsed():
    sec = int(time.time() - _T0)
    return f"{sec // 60:02d}:{sec % 60:02d}"


def progress(msg):
    if VERBOSE:
        print(f"[{_elapsed()}] {msg}", flush=True)


def progress_rows(name, df):
    if VERBOSE:
        progress(f"{name}：{len(df):,} 行")


def query_df(sql):
    """
    BigQuant DAI 对分区表（尤其 cn_stock_bar1d）要求通过 filters 指定分区范围。
    即使 SQL 中已经写了 WHERE date BETWEEN / IN，复杂 JOIN 场景下解析器也可能无法识别，
    因此统一传入 date filters，避免 Permission Error。
    """
    return dai.query(
        sql,
        filters={"date": [START_DATE, END_DATE]},
    ).df()


def try_query(sql):
    try:
        df = query_df(sql)
        if df is not None and len(df) > 0:
            return df
    except Exception as e:
        if VERBOSE:
            progress(f"SQL尝试失败：{str(e)[:140]}")
    return None


def first_success_query(sql_list, err_msg):
    last_error = None
    for i, sql in enumerate(sql_list, 1):
        try:
            df = query_df(sql)
            if df is not None and len(df) > 0:
                if VERBOSE:
                    progress(f"SQL第 {i} 个方案成功。")
                return df
        except Exception as e:
            last_error = e
            if FAST_QUERY_ONLY:
                break
            continue
    raise ValueError(f"{err_msg}。最后一次错误：{last_error}")


def date_in_sql(dates):
    return ", ".join([f"'{pd.to_datetime(x).strftime('%Y-%m-%d')}'" for x in dates])


def downcast_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce", downcast="float")
    return df


def to_date_str(x):
    return pd.to_datetime(x).strftime("%Y-%m-%d")


def calc_select_count(n, pct):
    if SELECT_COUNT_METHOD == "ceil":
        return max(1, int(np.ceil(n * pct)))
    return max(1, int(np.floor(n * pct)))


def winsorize_mad_array(x, k=WINSOR_K):
    x = np.asarray(x, dtype=np.float64)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    if not np.isfinite(mad) or mad <= 0:
        return x
    return np.clip(x, med - k * mad, med + k * mad)


def zscore_array(x):
    x = np.asarray(x, dtype=np.float64)
    mu = np.nanmean(x)
    sd = np.nanstd(x, ddof=1)
    if not np.isfinite(sd) or sd <= 0:
        return np.full(len(x), np.nan)
    return (x - mu) / sd


def normalize_industry_name(x):
    if pd.isna(x):
        return "UNKNOWN"
    s = str(x).strip()
    for token in [
        "申万一级行业", "申万一级", "申万", "SW2021", "SW", "中信一级行业", "中信一级",
        "一级行业", "行业", "（", "）", "(", ")", " ", "-", "_"
    ]:
        s = s.replace(token, "")
    alias = {
        "非银金融": "非银行金融",
        "非银行业金融": "非银行金融",
        "商业贸易": "商贸零售",
        "商贸零售业": "商贸零售",
        "食品饮料业": "食品饮料",
        "农林牧渔业": "农林牧渔",
        "轻工制造业": "轻工制造",
        "交通运输业": "交通运输",
        "建筑材料": "建材",
        "银行业": "银行",
        "通信设备": "通信",
        "医药生物": "医药",
    }
    return alias.get(s, s) if s else "UNKNOWN"


def is_bad_stock_name(name):
    txt = str(name).upper() if not pd.isna(name) else ""
    return ("ST" in txt) or ("退" in txt) or ("DELIST" in txt) or ("终止" in txt) or ("摘牌" in txt)


def get_current_date_from_engine(context, data):
    if data is not None and hasattr(data, "current_dt"):
        return pd.to_datetime(data.current_dt).strftime("%Y-%m-%d")
    for attr in ["current_dt", "now", "current_date"]:
        if hasattr(context, attr):
            v = getattr(context, attr)
            if v is not None:
                return pd.to_datetime(v).strftime("%Y-%m-%d")
    return None


def get_positions_dict(context):
    for method in ["get_account_positions", "get_positions"]:
        if hasattr(context, method):
            try:
                pos = getattr(context, method)()
                if pos is not None:
                    return pos
            except Exception:
                pass
    return {}


def position_amount(pos_obj):
    try:
        return float(getattr(pos_obj, "amount", 0))
    except Exception:
        try:
            return float(pos_obj.get("amount", 0))
        except Exception:
            return 0.0


def order_to_target_percent(context, instrument, weight):
    try:
        context.order_target_percent(instrument, float(weight))
        return True
    except Exception:
        try:
            context.order_percent(instrument, float(weight))
            return True
        except Exception as e:
            print(f"下单失败：{instrument}, target={weight:.6f}, err={e}", flush=True)
            return False


# =========================
# 3. 交易日和信号日
# =========================

def get_signal_dates():
    progress("开始获取交易日列表")

    sql = f"""
    SELECT DISTINCT date
    FROM cn_stock_bar1d
    WHERE date >= '{START_DATE}'
      AND date <= '{END_DATE}'
    ORDER BY date
    """
    trade_dates_df = query_df(sql)
    trade_dates_df["date"] = pd.to_datetime(trade_dates_df["date"])
    trade_dates = trade_dates_df["date"].drop_duplicates().sort_values().reset_index(drop=True)

    if len(trade_dates) < REBALANCE_DAYS + 2:
        raise ValueError("指定时间段内交易日过少，无法完成调仓回测。")

    raw_signal_dates = trade_dates.iloc[::REBALANCE_DAYS].tolist()

    signal_to_execution = {}
    for dt in raw_signal_dates:
        idx = trade_dates[trade_dates == dt].index
        if len(idx) == 0:
            continue
        next_idx = int(idx[0]) + 1
        if next_idx < len(trade_dates):
            signal_to_execution[to_date_str(dt)] = to_date_str(trade_dates.iloc[next_idx])

    signal_dates = [pd.to_datetime(k) for k in signal_to_execution.keys()]
    if len(signal_dates) == 0:
        raise ValueError("没有可用调仓信号日期。")

    progress(f"交易日数量：{len(trade_dates):,}；调仓信号截面数量：{len(signal_dates):,}")
    return signal_dates, signal_to_execution


# =========================
# 4. 一次性读取信号截面数据
# =========================

def query_signal_panel(signal_dates):
    progress("开始读取调仓截面数据：PB、市值、行业、证券简称")

    signal_date_sql = date_in_sql(signal_dates)

    sqls = [
        # fast path：常见 BigQuant 字段组合
        f"""
        SELECT
            v.date,
            v.instrument,
            v.pb,
            v.total_market_cap AS mkt_cap,
            p.cs_level1 AS industry,
            b.name AS stock_name
        FROM cn_stock_valuation v
        LEFT JOIN cn_stock_prefactors p
          ON v.date = p.date AND v.instrument = p.instrument
        LEFT JOIN (
            SELECT date, instrument, name
            FROM cn_stock_bar1d
            WHERE date IN ({signal_date_sql})
        ) b
          ON v.date = b.date AND v.instrument = b.instrument
        WHERE v.date IN ({signal_date_sql})
          AND v.pb IS NOT NULL
          AND v.total_market_cap IS NOT NULL
        """,
    ]

    if not FAST_QUERY_ONLY:
        sqls.extend([
            f"""
            SELECT
                v.date,
                v.instrument,
                v.pb,
                v.total_market_cap AS mkt_cap,
                p.industry AS industry,
                b.name AS stock_name
            FROM cn_stock_valuation v
            LEFT JOIN cn_stock_prefactors p
              ON v.date = p.date AND v.instrument = p.instrument
            LEFT JOIN (
                SELECT date, instrument, name
                FROM cn_stock_bar1d
                WHERE date IN ({signal_date_sql})
            ) b
              ON v.date = b.date AND v.instrument = b.instrument
            WHERE v.date IN ({signal_date_sql})
              AND v.pb IS NOT NULL
              AND v.total_market_cap IS NOT NULL
            """,
            f"""
            SELECT
                v.date,
                v.instrument,
                v.pb,
                v.total_mv AS mkt_cap,
                p.cs_level1 AS industry,
                b.name AS stock_name
            FROM cn_stock_valuation v
            LEFT JOIN cn_stock_prefactors p
              ON v.date = p.date AND v.instrument = p.instrument
            LEFT JOIN (
                SELECT date, instrument, name
                FROM cn_stock_bar1d
                WHERE date IN ({signal_date_sql})
            ) b
              ON v.date = b.date AND v.instrument = b.instrument
            WHERE v.date IN ({signal_date_sql})
              AND v.pb IS NOT NULL
              AND v.total_mv IS NOT NULL
            """,
            f"""
            SELECT
                v.date,
                v.instrument,
                v.pb,
                v.total_market_cap AS mkt_cap,
                'UNKNOWN' AS industry,
                b.name AS stock_name
            FROM cn_stock_valuation v
            LEFT JOIN (
                SELECT date, instrument, name
                FROM cn_stock_bar1d
                WHERE date IN ({signal_date_sql})
            ) b
              ON v.date = b.date AND v.instrument = b.instrument
            WHERE v.date IN ({signal_date_sql})
              AND v.pb IS NOT NULL
              AND v.total_market_cap IS NOT NULL
            """,
        ])

    df = first_success_query(sqls, "未能读取信号截面所需字段")
    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["industry"] = df["industry"].fillna("UNKNOWN").map(normalize_industry_name).astype("category")
    df["stock_name"] = df["stock_name"].fillna("").astype(str)

    downcast_numeric(df, ["pb", "mkt_cap"])
    df = df.dropna(subset=["date", "instrument", "pb", "mkt_cap"])
    df = df[(df["pb"] > 0) & (df["mkt_cap"] > 0)].copy()

    # 信号日剔除可识别 ST、*ST、退市；只使用信号日证券简称，不使用未来状态。
    before = len(df)
    bad_name = df["stock_name"].map(is_bad_stock_name).astype(bool)
    df = df[~bad_name].copy()
    progress(f"证券简称过滤 ST/退市：剔除 {before - len(df):,} 行；剩余 {len(df):,} 行")

    df[FACTOR_NAME] = 1.0 / df["pb"].astype(float)
    df = df[np.isfinite(df[FACTOR_NAME])].copy()
    downcast_numeric(df, [FACTOR_NAME])

    df = df.sort_values(["date", "instrument"], kind="mergesort")
    df = df.drop_duplicates(subset=["date", "instrument"], keep="last")

    # 只保留后续需要的列
    df = df[["date", "instrument", FACTOR_NAME, "mkt_cap", "industry"]].reset_index(drop=True)
    progress_rows("优化后信号截面数据", df)
    return df


# =========================
# 5. 截面中性化、分层、选股
# =========================

def neutralize_bp_group_and_select(g):
    g = g[["date", "instrument", FACTOR_NAME, "mkt_cap", "industry"]].copy()
    g = g.sort_values(["instrument"], kind="mergesort").reset_index(drop=True)

    if len(g) < max(MIN_OBS_PER_CROSS_SECTION, N_SIZE_GROUPS * MIN_STOCKS_PER_SIZE_GROUP // 2):
        return None

    y0 = pd.to_numeric(g[FACTOR_NAME], errors="coerce")
    cap = pd.to_numeric(g["mkt_cap"], errors="coerce")
    valid0 = y0.notna() & cap.notna() & (cap > 0)
    if valid0.sum() < MIN_OBS_PER_CROSS_SECTION:
        return None

    g = g.loc[valid0].copy()
    g["log_mkt_z"] = zscore_array(np.log(g["mkt_cap"].astype(float).values))
    g = g.dropna(subset=[FACTOR_NAME, "log_mkt_z"])
    if len(g) < MIN_OBS_PER_CROSS_SECTION:
        return None

    # FWL：先按行业去均值，再对市值残差做一元回归。
    ind_group = g.groupby("industry", sort=True, observed=True)
    y_dm = g[FACTOR_NAME].astype(float) - ind_group[FACTOR_NAME].transform("mean").astype(float)
    x_dm = g["log_mkt_z"].astype(float) - ind_group["log_mkt_z"].transform("mean").astype(float)

    y = y_dm.to_numpy(dtype=np.float64)
    x = x_dm.to_numpy(dtype=np.float64)
    valid = np.isfinite(y) & np.isfinite(x)

    if valid.sum() < MIN_OBS_PER_CROSS_SECTION:
        return None

    x_valid = x[valid]
    y_valid = y[valid]
    x_var = float(np.sum(x_valid * x_valid))
    if x_var <= 0 or not np.isfinite(x_var):
        return None

    beta = float(np.sum(x_valid * y_valid) / x_var)
    resid = y_valid - beta * x_valid

    out = g.loc[valid, ["date", "instrument", "mkt_cap"]].copy()
    resid_w = winsorize_mad_array(resid, WINSOR_K)
    out[NEUTRAL_FACTOR_NAME] = zscore_array(resid_w)
    out = out.dropna(subset=[NEUTRAL_FACTOR_NAME, "mkt_cap"])

    if len(out) < MIN_OBS_PER_CROSS_SECTION:
        return None

    # 市值15组，排序固定，保证复现稳定。
    out = out.sort_values(["mkt_cap", "instrument"], ascending=[True, True], kind="mergesort").reset_index(drop=True)
    rank = out["mkt_cap"].rank(method="first", ascending=True)

    try:
        out["size_group"] = pd.qcut(
            rank,
            q=N_SIZE_GROUPS,
            labels=list(range(1, N_SIZE_GROUPS + 1))
        ).astype(int)
    except Exception:
        return None

    selected_parts = []
    for size_group in SIZE_GROUPS_TO_TRADE:
        sg = out[out["size_group"] == size_group].copy()
        if len(sg) < MIN_STOCKS_PER_SIZE_GROUP:
            continue

        n_select = calc_select_count(len(sg), TOP_PCT)
        sg = sg.sort_values([NEUTRAL_FACTOR_NAME, "instrument"], ascending=[False, True], kind="mergesort")
        selected_parts.append(sg.head(n_select))

    if not selected_parts:
        return None

    selected = pd.concat(selected_parts, ignore_index=True)

    if GROUP_CAPITAL_EQUAL:
        layer_count = selected["size_group"].nunique()
        selected["group_count"] = selected.groupby("size_group")["instrument"].transform("count")
        selected["target_weight"] = 1.0 / layer_count / selected["group_count"]
    else:
        selected["target_weight"] = 1.0 / len(selected)

    return selected[["date", "instrument", "size_group", NEUTRAL_FACTOR_NAME, "target_weight"]]


def build_signal_df(signal_panel, signal_to_execution):
    progress("开始逐截面中性化、15档市值分组与选股")

    selected_parts = []
    all_signal_dates = sorted(signal_panel["date"].drop_duplicates())

    for i, dt in enumerate(all_signal_dates, 1):
        g = signal_panel.loc[signal_panel["date"] == dt]
        if VERBOSE and (i == 1 or i % 5 == 0 or i == len(all_signal_dates)):
            progress(f"选股进度：{i}/{len(all_signal_dates)}，信号日 {to_date_str(dt)}，样本 {len(g):,}")

        sel = neutralize_bp_group_and_select(g)
        if sel is not None and len(sel) > 0:
            selected_parts.append(sel)

    if not selected_parts:
        raise ValueError("没有形成任何有效选股结果，请检查参数或数据。")

    selected_df = pd.concat(selected_parts, ignore_index=True)
    selected_df["signal_date"] = selected_df["date"].dt.strftime("%Y-%m-%d")
    selected_df["execution_date"] = selected_df["signal_date"].map(signal_to_execution)
    selected_df = selected_df.dropna(subset=["execution_date"]).copy()

    signal_df = selected_df[[
        "signal_date", "execution_date", "instrument", "size_group", NEUTRAL_FACTOR_NAME, "target_weight"
    ]].copy()

    signal_df = signal_df.rename(columns={"signal_date": "date"})
    signal_df["date"] = pd.to_datetime(signal_df["date"]).dt.strftime("%Y-%m-%d")
    signal_df["instrument"] = signal_df["instrument"].astype(str)

    signal_df = signal_df.sort_values(
        ["date", "size_group", NEUTRAL_FACTOR_NAME, "instrument"],
        ascending=[True, True, False, True],
        kind="mergesort"
    ).reset_index(drop=True)

    progress_rows("最终交易信号", signal_df)

    # 默认不展示中间交易信号摘要表，避免干扰正式回测输出。
    # 若需要调试选股数量和权重，可将 DISPLAY_SIGNAL_SUMMARY 改为 True。
    if DISPLAY_SIGNAL_SUMMARY:
        signal_summary = (
            signal_df.groupby("date")
            .agg(
                execution_date=("execution_date", "first"),
                stock_count=("instrument", "count"),
                avg_weight=("target_weight", "mean"),
                min_weight=("target_weight", "min"),
                max_weight=("target_weight", "max"),
            )
            .reset_index()
        )
        progress("交易信号摘要：")
        display(signal_summary.head(20))

    return signal_df


# =========================
# 6. BigTrader 原生回测
# =========================

def make_engine_inputs(signal_df):
    backtest_data = signal_df[["date", "instrument", "target_weight"]].copy()
    backtest_data["date"] = pd.to_datetime(backtest_data["date"]).dt.strftime("%Y-%m-%d")
    backtest_data["instrument"] = backtest_data["instrument"].astype(str)

    signal_by_date = {
        d: g[["instrument", "target_weight"]].copy()
        for d, g in signal_df.groupby("date", sort=True)
    }

    target_by_date = {
        d: set(g["instrument"].astype(str))
        for d, g in signal_df.groupby("date", sort=True)
    }
    return backtest_data, signal_by_date, target_by_date


def make_callbacks(backtest_data, signal_by_date, target_by_date):
    def initialize(context):
        try:
            context.set_commission(
                bigtrader.PerOrder(
                    buy_cost=BUY_COST,
                    sell_cost=SELL_COST,
                    min_cost=MIN_COMMISSION,
                )
            )
        except Exception as e:
            print(f"设置手续费失败，将使用引擎默认费率。原因：{e}", flush=True)

        context.signal_by_date = signal_by_date
        context.target_by_date = target_by_date
        context.rebalance_dates = set(signal_by_date.keys())

        try:
            context.subscribe_bar(list(backtest_data["instrument"].drop_duplicates()), "1d", None)
        except Exception:
            pass

    def handle_data(context, data):
        current_date = get_current_date_from_engine(context, data)
        if current_date is None:
            return

        if current_date not in context.rebalance_dates:
            return

        today_signal = context.signal_by_date.get(current_date)
        if today_signal is None or len(today_signal) == 0:
            return

        target_weights = dict(zip(today_signal["instrument"].astype(str), today_signal["target_weight"].astype(float)))
        target_instruments = set(target_weights.keys())

        positions = get_positions_dict(context)
        holding_instruments = set()
        for ins, pos in positions.items():
            if position_amount(pos) > 0:
                holding_instruments.add(str(ins))

        # 卖出不在目标池中的股票；是否能成交由 BigTrader 按实际撮合日状态处理。
        for ins in sorted(holding_instruments - target_instruments):
            order_to_target_percent(context, ins, 0.0)

        # 买入或调整目标股票到目标权重。
        for ins in sorted(target_weights.keys()):
            order_to_target_percent(context, ins, target_weights[ins])

    return initialize, handle_data


def run_bigtrader(backtest_data, signal_by_date, target_by_date):
    initialize, handle_data = make_callbacks(backtest_data, signal_by_date, target_by_date)

    run_kwargs = dict(
        data=backtest_data,
        start_date=START_DATE,
        end_date=END_DATE,
        initialize=initialize,
        handle_data=handle_data,
        capital_base=CAPITAL_BASE,
        benchmark=BENCHMARK,
    )

    try:
        run_kwargs["market"] = bigtrader.Market.CN_STOCK
    except Exception:
        pass

    try:
        run_kwargs["frequency"] = bigtrader.Frequency.DAILY
    except Exception:
        run_kwargs["frequency"] = "1d"

    return bigtrader.run(**run_kwargs)


# =========================
# 7. 主流程
# =========================

def main():
    progress("开始执行性能优化版 BP 分层策略")

    signal_dates, signal_to_execution = get_signal_dates()
    signal_panel = query_signal_panel(signal_dates)
    signal_df = build_signal_df(signal_panel, signal_to_execution)

    # 释放大表，减少 BigTrader 阶段内存压力
    del signal_panel
    gc.collect()

    backtest_data, signal_by_date, target_by_date = make_engine_inputs(signal_df)

    progress("开始运行 BigTrader 原生回测")
    performance = run_bigtrader(backtest_data, signal_by_date, target_by_date)
    progress("BigTrader 回测完成")

    try:
        display(performance.summary)
    except Exception:
        display(performance)

    return performance, signal_df, backtest_data


performance, signal_df, backtest_data = main()


从输出的结果来看，该因子同样在小盘股当中能获得有利表现，并且值得注意的是，该因子构造简单，但却能够获得72%的胜率，0.71的夏普，非常值得后期继续加入防御性+收益补偿策略来优化其表现

## 防御性+收益补偿策略

In [ ]:
# -*- coding: utf-8 -*-
"""
BP 因子：市值行业中性化后按市值分层选股回测（防御性+收益补偿，性能优化版）

相对上一版的优化点
----------------
1. 合并数据查询与 DAI 分区过滤：
   - 所有 DAI 查询统一传入 filters={"date": [START_DATE, END_DATE]}，避免 cn_stock_bar1d 分区权限错误；
   - 不再分别查询 PB/市值、行业、状态后多次 merge；
   - 默认一次 SQL 读取信号日所需字段：PB、市值、行业、证券简称。
2. 减少兼容性探测：
   - 默认走 BigQuant 常用字段 fast path；
   - 只在 fast path 失败时尝试少量备用 SQL。
3. 减少内存占用：
   - 只读取调仓信号日截面，不读取全量日频面板；
   - 只保留必要列；
   - 数值列 downcast 到 float32；
   - 行业列转 category。
4. 优化中性化：
   - 使用 FWL 方式剔除行业固定效应和市值暴露；
   - 避免为每个截面构造大规模行业 dummy 矩阵。
5. 减少日志和中间展示：
   - VERBOSE 控制进度输出；
   - 不主动保存任何结果文件。

策略逻辑
--------
- BP = 1 / PB；
- 信号日剔除可识别 ST、*ST、退市股票；
- 每个截面做：市值 + 行业中性化 -> MAD去极值 -> Z-score标准化；
- 全市场按总市值从小到大分为 15 组；
- 通过 SIZE_GROUPS_TO_TRADE 指定交易市值组；
- 每个指定市值组内，买入因子值最大的前 10%；
- 调仓周期 60 个交易日；
- 交易撮合、涨跌停、停牌由 BigTrader 原生回测引擎处理；
- 加入中证1000指数趋势过滤：中证1000跌破60日均线时，因子股票仓位降至10%，释放仓位等权买入工商银行、交通银行、中国银行；
-重新站上60日均线后恢复原因子持仓；
- 趋势仓位每日检查，趋势变化时只调整仓位，不重新选股；
- 不读取下一交易日涨跌停、停牌、成交量等信息，避免未来函数。
"""

import warnings
warnings.filterwarnings("ignore")

import gc
import time
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

import dai
from bigquant import bigtrader


# =========================
# 1. 参数设置
# =========================

START_DATE = "2022-01-01"
END_DATE = "2026-06-30"

FACTOR_NAME = "bp_raw"
NEUTRAL_FACTOR_NAME = "bp_mkt_ind_neutral"

# 1=最小市值组，15=最大市值组
SIZE_GROUPS_TO_TRADE = [1,2]
N_SIZE_GROUPS = 15

TOP_PCT = 0.15
REBALANCE_DAYS = 60

SELECT_COUNT_METHOD = "floor"      # "floor" or "ceil"
GROUP_CAPITAL_EQUAL = True         # True：市值组等资金；False：全部入选股票整体等权

WINSOR_K = 5
MIN_OBS_PER_CROSS_SECTION = 100
MIN_STOCKS_PER_SIZE_GROUP = 20

CAPITAL_BASE = 1_000_000
BENCHMARK = "000300.SH"

BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COMMISSION = 5

# =========================
# 防御性 + 收益补偿参数
# =========================

USE_DEFENSIVE_COMPENSATION = True

# 中证1000指数。不同 BigQuant 环境中指数代码可能有差异，代码中会尝试多个候选。
TREND_INDEX_CODE = "000852.SH"
TREND_INDEX_CANDIDATES = ["000852.SH", "000852.CSI", "000852.XSHG"]

TREND_MA_WINDOW = 60

# 严谨性：
# True 表示 t 日调仓使用 t-1 日收盘后已经可知的趋势状态；
# 这样即使 BigTrader 在 t 日开盘撮合，也不会读取 t 日收盘后的信息。
TREND_USE_PREVIOUS_TRADING_DAY = True

# 风险开启：因子组合满仓；风险关闭：因子股票保留10%，其余90%买入三只银行股。
RISK_ON_STOCK_EXPOSURE = 1.00
RISK_OFF_STOCK_EXPOSURE = 0.10
RISK_ON_DEFENSIVE_EXPOSURE = 0.00
RISK_OFF_DEFENSIVE_EXPOSURE = RISK_ON_STOCK_EXPOSURE - RISK_OFF_STOCK_EXPOSURE

DEFENSIVE_BANK_ASSETS = {
    "601398.SH": "工商银行",
    "601328.SH": "交通银行",
    "601988.SH": "中国银行",
}

# 查询均线需要向前取历史指数数据。
TREND_QUERY_START_DATE = (
    pd.to_datetime(START_DATE) - pd.Timedelta(days=max(260, TREND_MA_WINDOW * 4))
).strftime("%Y-%m-%d")

# 趋势仓位切换日志。False 可进一步减少日志输出。
PRINT_TREND_SWITCH_LOG = True

# 性能开关
VERBOSE = True
DISPLAY_SIGNAL_SUMMARY = False          # False：不打印中间交易信号摘要表
FAST_QUERY_ONLY = False             # True：只走最快字段组合；False：fast path失败后尝试少量备用字段


START_DATE = pd.to_datetime(START_DATE).strftime("%Y-%m-%d")
END_DATE = pd.to_datetime(END_DATE).strftime("%Y-%m-%d")

if not SIZE_GROUPS_TO_TRADE:
    raise ValueError("SIZE_GROUPS_TO_TRADE 不能为空。")

bad_groups = [g for g in SIZE_GROUPS_TO_TRADE if int(g) < 1 or int(g) > N_SIZE_GROUPS]
if bad_groups:
    raise ValueError(f"SIZE_GROUPS_TO_TRADE 中存在非法市值组：{bad_groups}，有效范围为 1~{N_SIZE_GROUPS}。")

SIZE_GROUPS_TO_TRADE = sorted(set([int(x) for x in SIZE_GROUPS_TO_TRADE]))

if SELECT_COUNT_METHOD not in ["floor", "ceil"]:
    raise ValueError("SELECT_COUNT_METHOD 只能是 'floor' 或 'ceil'。")


# =========================
# 2. 工具函数
# =========================

_T0 = time.time()


def _elapsed():
    sec = int(time.time() - _T0)
    return f"{sec // 60:02d}:{sec % 60:02d}"


def progress(msg):
    if VERBOSE:
        print(f"[{_elapsed()}] {msg}", flush=True)


def progress_rows(name, df):
    if VERBOSE:
        progress(f"{name}：{len(df):,} 行")


def query_df(sql, start_date=None, end_date=None):
    """
    BigQuant DAI 对分区表（尤其 cn_stock_bar1d）要求通过 filters 指定分区范围。
    即使 SQL 中已经写了 WHERE date BETWEEN / IN，复杂 JOIN 场景下解析器也可能无法识别，
    因此统一传入 date filters，避免 Permission Error。

    趋势均线需要读取 START_DATE 之前的数据，因此这里允许传入更早的 start_date。
    """
    fs = START_DATE if start_date is None else pd.to_datetime(start_date).strftime("%Y-%m-%d")
    fe = END_DATE if end_date is None else pd.to_datetime(end_date).strftime("%Y-%m-%d")
    return dai.query(
        sql,
        filters={"date": [fs, fe]},
    ).df()


def try_query(sql, start_date=None, end_date=None):
    try:
        df = query_df(sql, start_date=start_date, end_date=end_date)
        if df is not None and len(df) > 0:
            return df
    except Exception as e:
        if VERBOSE:
            progress(f"SQL尝试失败：{str(e)[:140]}")
    return None


def first_success_query(sql_list, err_msg, start_date=None, end_date=None):
    last_error = None
    for i, sql in enumerate(sql_list, 1):
        try:
            df = query_df(sql, start_date=start_date, end_date=end_date)
            if df is not None and len(df) > 0:
                if VERBOSE:
                    progress(f"SQL第 {i} 个方案成功。")
                return df
        except Exception as e:
            last_error = e
            if FAST_QUERY_ONLY:
                break
            continue
    raise ValueError(f"{err_msg}。最后一次错误：{last_error}")


def date_in_sql(dates):
    return ", ".join([f"'{pd.to_datetime(x).strftime('%Y-%m-%d')}'" for x in dates])


def downcast_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce", downcast="float")
    return df


def to_date_str(x):
    return pd.to_datetime(x).strftime("%Y-%m-%d")


def calc_select_count(n, pct):
    if SELECT_COUNT_METHOD == "ceil":
        return max(1, int(np.ceil(n * pct)))
    return max(1, int(np.floor(n * pct)))


def winsorize_mad_array(x, k=WINSOR_K):
    x = np.asarray(x, dtype=np.float64)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    if not np.isfinite(mad) or mad <= 0:
        return x
    return np.clip(x, med - k * mad, med + k * mad)


def zscore_array(x):
    x = np.asarray(x, dtype=np.float64)
    mu = np.nanmean(x)
    sd = np.nanstd(x, ddof=1)
    if not np.isfinite(sd) or sd <= 0:
        return np.full(len(x), np.nan)
    return (x - mu) / sd


def normalize_industry_name(x):
    if pd.isna(x):
        return "UNKNOWN"
    s = str(x).strip()
    for token in [
        "申万一级行业", "申万一级", "申万", "SW2021", "SW", "中信一级行业", "中信一级",
        "一级行业", "行业", "（", "）", "(", ")", " ", "-", "_"
    ]:
        s = s.replace(token, "")
    alias = {
        "非银金融": "非银行金融",
        "非银行业金融": "非银行金融",
        "商业贸易": "商贸零售",
        "商贸零售业": "商贸零售",
        "食品饮料业": "食品饮料",
        "农林牧渔业": "农林牧渔",
        "轻工制造业": "轻工制造",
        "交通运输业": "交通运输",
        "建筑材料": "建材",
        "银行业": "银行",
        "通信设备": "通信",
        "医药生物": "医药",
    }
    return alias.get(s, s) if s else "UNKNOWN"


def is_bad_stock_name(name):
    txt = str(name).upper() if not pd.isna(name) else ""
    return ("ST" in txt) or ("退" in txt) or ("DELIST" in txt) or ("终止" in txt) or ("摘牌" in txt)


def get_current_date_from_engine(context, data):
    if data is not None and hasattr(data, "current_dt"):
        return pd.to_datetime(data.current_dt).strftime("%Y-%m-%d")
    for attr in ["current_dt", "now", "current_date"]:
        if hasattr(context, attr):
            v = getattr(context, attr)
            if v is not None:
                return pd.to_datetime(v).strftime("%Y-%m-%d")
    return None


def get_positions_dict(context):
    for method in ["get_account_positions", "get_positions"]:
        if hasattr(context, method):
            try:
                pos = getattr(context, method)()
                if pos is not None:
                    return pos
            except Exception:
                pass
    return {}


def position_amount(pos_obj):
    try:
        return float(getattr(pos_obj, "amount", 0))
    except Exception:
        try:
            return float(pos_obj.get("amount", 0))
        except Exception:
            return 0.0


def order_to_target_percent(context, instrument, weight):
    try:
        context.order_target_percent(instrument, float(weight))
        return True
    except Exception:
        try:
            context.order_percent(instrument, float(weight))
            return True
        except Exception as e:
            print(f"下单失败：{instrument}, target={weight:.6f}, err={e}", flush=True)
            return False



def to_bigtrader_instrument(inst):
    """
    兼容不同 BigQuant 环境中的证券代码后缀。
    """
    s = str(inst)
    if s.endswith(".SZA"):
        return s[:-4] + ".SZ"
    if s.endswith(".SHA"):
        return s[:-4] + ".SH"
    if s.endswith(".BJA"):
        return s[:-4] + ".BJ"
    return s


def build_trend_allocation_df(trade_dates):
    """
    构造每日趋势仓位序列。

    规则：
    - 中证1000收盘价 < 60日均线：因子股票仓位降至10%，90%等权买入三只银行股；
    - 中证1000收盘价 >= 60日均线：恢复原因子持仓；
    - 默认 t 日使用 t-1 交易日趋势状态，避免开盘调仓读取 t 日收盘价。
    """
    trade_dates = pd.DatetimeIndex(pd.to_datetime(trade_dates)).sort_values()
    if len(trade_dates) == 0:
        raise ValueError("trade_dates 为空，无法构造趋势仓位序列。")

    if not USE_DEFENSIVE_COMPENSATION:
        return pd.DataFrame(
            {
                "stock_exposure": float(RISK_ON_STOCK_EXPOSURE),
                "defensive_exposure": float(RISK_ON_DEFENSIVE_EXPOSURE),
                "risk_on": True,
            },
            index=trade_dates,
        )

    sqls = []
    used_codes = list(dict.fromkeys([TREND_INDEX_CODE] + TREND_INDEX_CANDIDATES))
    for code in used_codes:
        sqls.append(
            f"""
            SELECT date, instrument, close
            FROM cn_stock_index_bar1d
            WHERE instrument = '{code}'
              AND date >= '{TREND_QUERY_START_DATE}'
              AND date <= '{END_DATE}'
            ORDER BY date
            """
        )
        sqls.append(
            f"""
            SELECT date, instrument, close
            FROM cn_stock_bar1d
            WHERE instrument = '{code}'
              AND date >= '{TREND_QUERY_START_DATE}'
              AND date <= '{END_DATE}'
            ORDER BY date
            """
        )

    idx = first_success_query(
        sqls,
        f"无法读取趋势过滤指数 {TREND_INDEX_CODE} 的日线收盘价",
        start_date=TREND_QUERY_START_DATE,
        end_date=END_DATE,
    )

    idx["date"] = pd.to_datetime(idx["date"])
    idx["close"] = pd.to_numeric(idx["close"], errors="coerce")
    idx = idx.dropna(subset=["date", "close"])
    idx = idx[idx["close"] > 0].copy()
    idx = idx.sort_values("date", kind="mergesort").drop_duplicates("date", keep="last")

    if len(idx) < TREND_MA_WINDOW:
        raise ValueError(f"指数数据不足，无法计算 {TREND_MA_WINDOW} 日均线。")

    close = idx.set_index("date")["close"].sort_index()
    ma = close.rolling(window=TREND_MA_WINDOW, min_periods=TREND_MA_WINDOW).mean()

    # 跌破均线：risk_off；站上或等于均线：risk_on。
    risk_on_raw = close >= ma
    risk_on_raw = risk_on_raw.where(ma.notna(), True)

    raw_alloc = pd.DataFrame(
        {
            "risk_on_raw": risk_on_raw.astype(bool),
            "stock_exposure_raw": np.where(risk_on_raw, RISK_ON_STOCK_EXPOSURE, RISK_OFF_STOCK_EXPOSURE),
            "defensive_exposure_raw": np.where(risk_on_raw, RISK_ON_DEFENSIVE_EXPOSURE, RISK_OFF_DEFENSIVE_EXPOSURE),
        },
        index=close.index,
    )

    raw_alloc = raw_alloc.reindex(trade_dates).ffill()

    if TREND_USE_PREVIOUS_TRADING_DAY:
        risk_on = raw_alloc["risk_on_raw"].shift(1).fillna(True).astype(bool)
        stock_exposure = raw_alloc["stock_exposure_raw"].shift(1).fillna(float(RISK_ON_STOCK_EXPOSURE))
        defensive_exposure = raw_alloc["defensive_exposure_raw"].shift(1).fillna(float(RISK_ON_DEFENSIVE_EXPOSURE))
    else:
        risk_on = raw_alloc["risk_on_raw"].fillna(True).astype(bool)
        stock_exposure = raw_alloc["stock_exposure_raw"].fillna(float(RISK_ON_STOCK_EXPOSURE))
        defensive_exposure = raw_alloc["defensive_exposure_raw"].fillna(float(RISK_ON_DEFENSIVE_EXPOSURE))

    out = pd.DataFrame(
        {
            "stock_exposure": stock_exposure.astype(float),
            "defensive_exposure": defensive_exposure.astype(float),
            "risk_on": risk_on.astype(bool),
        },
        index=trade_dates,
    )
    return out


# =========================
# 3. 交易日、信号日和趋势仓位
# =========================

def get_signal_dates():
    progress("开始获取交易日列表")

    sql = f"""
    SELECT DISTINCT date
    FROM cn_stock_bar1d
    WHERE date >= '{START_DATE}'
      AND date <= '{END_DATE}'
    ORDER BY date
    """
    trade_dates_df = query_df(sql)
    trade_dates_df["date"] = pd.to_datetime(trade_dates_df["date"])
    trade_dates = trade_dates_df["date"].drop_duplicates().sort_values().reset_index(drop=True)

    if len(trade_dates) < REBALANCE_DAYS + 2:
        raise ValueError("指定时间段内交易日过少，无法完成调仓回测。")

    raw_signal_dates = trade_dates.iloc[::REBALANCE_DAYS].tolist()

    signal_to_execution = {}
    for dt in raw_signal_dates:
        idx = trade_dates[trade_dates == dt].index
        if len(idx) == 0:
            continue
        next_idx = int(idx[0]) + 1
        if next_idx < len(trade_dates):
            signal_to_execution[to_date_str(dt)] = to_date_str(trade_dates.iloc[next_idx])

    signal_dates = [pd.to_datetime(k) for k in signal_to_execution.keys()]
    if len(signal_dates) == 0:
        raise ValueError("没有可用调仓信号日期。")

    progress(f"交易日数量：{len(trade_dates):,}；调仓信号截面数量：{len(signal_dates):,}")
    return trade_dates, signal_dates, signal_to_execution


# =========================
# 4. 一次性读取信号截面数据
# =========================

def query_signal_panel(signal_dates):
    progress("开始读取调仓截面数据：PB、市值、行业、证券简称")

    signal_date_sql = date_in_sql(signal_dates)

    sqls = [
        # fast path：常见 BigQuant 字段组合
        f"""
        SELECT
            v.date,
            v.instrument,
            v.pb,
            v.total_market_cap AS mkt_cap,
            p.cs_level1 AS industry,
            b.name AS stock_name
        FROM cn_stock_valuation v
        LEFT JOIN cn_stock_prefactors p
          ON v.date = p.date AND v.instrument = p.instrument
        LEFT JOIN (
            SELECT date, instrument, name
            FROM cn_stock_bar1d
            WHERE date IN ({signal_date_sql})
        ) b
          ON v.date = b.date AND v.instrument = b.instrument
        WHERE v.date IN ({signal_date_sql})
          AND v.pb IS NOT NULL
          AND v.total_market_cap IS NOT NULL
        """,
    ]

    if not FAST_QUERY_ONLY:
        sqls.extend([
            f"""
            SELECT
                v.date,
                v.instrument,
                v.pb,
                v.total_market_cap AS mkt_cap,
                p.industry AS industry,
                b.name AS stock_name
            FROM cn_stock_valuation v
            LEFT JOIN cn_stock_prefactors p
              ON v.date = p.date AND v.instrument = p.instrument
            LEFT JOIN (
                SELECT date, instrument, name
                FROM cn_stock_bar1d
                WHERE date IN ({signal_date_sql})
            ) b
              ON v.date = b.date AND v.instrument = b.instrument
            WHERE v.date IN ({signal_date_sql})
              AND v.pb IS NOT NULL
              AND v.total_market_cap IS NOT NULL
            """,
            f"""
            SELECT
                v.date,
                v.instrument,
                v.pb,
                v.total_mv AS mkt_cap,
                p.cs_level1 AS industry,
                b.name AS stock_name
            FROM cn_stock_valuation v
            LEFT JOIN cn_stock_prefactors p
              ON v.date = p.date AND v.instrument = p.instrument
            LEFT JOIN (
                SELECT date, instrument, name
                FROM cn_stock_bar1d
                WHERE date IN ({signal_date_sql})
            ) b
              ON v.date = b.date AND v.instrument = b.instrument
            WHERE v.date IN ({signal_date_sql})
              AND v.pb IS NOT NULL
              AND v.total_mv IS NOT NULL
            """,
            f"""
            SELECT
                v.date,
                v.instrument,
                v.pb,
                v.total_market_cap AS mkt_cap,
                'UNKNOWN' AS industry,
                b.name AS stock_name
            FROM cn_stock_valuation v
            LEFT JOIN (
                SELECT date, instrument, name
                FROM cn_stock_bar1d
                WHERE date IN ({signal_date_sql})
            ) b
              ON v.date = b.date AND v.instrument = b.instrument
            WHERE v.date IN ({signal_date_sql})
              AND v.pb IS NOT NULL
              AND v.total_market_cap IS NOT NULL
            """,
        ])

    df = first_success_query(sqls, "未能读取信号截面所需字段")
    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["industry"] = df["industry"].fillna("UNKNOWN").map(normalize_industry_name).astype("category")
    df["stock_name"] = df["stock_name"].fillna("").astype(str)

    downcast_numeric(df, ["pb", "mkt_cap"])
    df = df.dropna(subset=["date", "instrument", "pb", "mkt_cap"])
    df = df[(df["pb"] > 0) & (df["mkt_cap"] > 0)].copy()

    # 信号日剔除可识别 ST、*ST、退市；只使用信号日证券简称，不使用未来状态。
    before = len(df)
    bad_name = df["stock_name"].map(is_bad_stock_name).astype(bool)
    df = df[~bad_name].copy()
    progress(f"证券简称过滤 ST/退市：剔除 {before - len(df):,} 行；剩余 {len(df):,} 行")

    df[FACTOR_NAME] = 1.0 / df["pb"].astype(float)
    df = df[np.isfinite(df[FACTOR_NAME])].copy()
    downcast_numeric(df, [FACTOR_NAME])

    df = df.sort_values(["date", "instrument"], kind="mergesort")
    df = df.drop_duplicates(subset=["date", "instrument"], keep="last")

    # 只保留后续需要的列
    df = df[["date", "instrument", FACTOR_NAME, "mkt_cap", "industry"]].reset_index(drop=True)
    progress_rows("优化后信号截面数据", df)
    return df


# =========================
# 5. 截面中性化、分层、选股
# =========================

def neutralize_bp_group_and_select(g):
    g = g[["date", "instrument", FACTOR_NAME, "mkt_cap", "industry"]].copy()
    g = g.sort_values(["instrument"], kind="mergesort").reset_index(drop=True)

    if len(g) < max(MIN_OBS_PER_CROSS_SECTION, N_SIZE_GROUPS * MIN_STOCKS_PER_SIZE_GROUP // 2):
        return None

    y0 = pd.to_numeric(g[FACTOR_NAME], errors="coerce")
    cap = pd.to_numeric(g["mkt_cap"], errors="coerce")
    valid0 = y0.notna() & cap.notna() & (cap > 0)
    if valid0.sum() < MIN_OBS_PER_CROSS_SECTION:
        return None

    g = g.loc[valid0].copy()
    g["log_mkt_z"] = zscore_array(np.log(g["mkt_cap"].astype(float).values))
    g = g.dropna(subset=[FACTOR_NAME, "log_mkt_z"])
    if len(g) < MIN_OBS_PER_CROSS_SECTION:
        return None

    # FWL：先按行业去均值，再对市值残差做一元回归。
    ind_group = g.groupby("industry", sort=True, observed=True)
    y_dm = g[FACTOR_NAME].astype(float) - ind_group[FACTOR_NAME].transform("mean").astype(float)
    x_dm = g["log_mkt_z"].astype(float) - ind_group["log_mkt_z"].transform("mean").astype(float)

    y = y_dm.to_numpy(dtype=np.float64)
    x = x_dm.to_numpy(dtype=np.float64)
    valid = np.isfinite(y) & np.isfinite(x)

    if valid.sum() < MIN_OBS_PER_CROSS_SECTION:
        return None

    x_valid = x[valid]
    y_valid = y[valid]
    x_var = float(np.sum(x_valid * x_valid))
    if x_var <= 0 or not np.isfinite(x_var):
        return None

    beta = float(np.sum(x_valid * y_valid) / x_var)
    resid = y_valid - beta * x_valid

    out = g.loc[valid, ["date", "instrument", "mkt_cap"]].copy()
    resid_w = winsorize_mad_array(resid, WINSOR_K)
    out[NEUTRAL_FACTOR_NAME] = zscore_array(resid_w)
    out = out.dropna(subset=[NEUTRAL_FACTOR_NAME, "mkt_cap"])

    if len(out) < MIN_OBS_PER_CROSS_SECTION:
        return None

    # 市值15组，排序固定，保证复现稳定。
    out = out.sort_values(["mkt_cap", "instrument"], ascending=[True, True], kind="mergesort").reset_index(drop=True)
    rank = out["mkt_cap"].rank(method="first", ascending=True)

    try:
        out["size_group"] = pd.qcut(
            rank,
            q=N_SIZE_GROUPS,
            labels=list(range(1, N_SIZE_GROUPS + 1))
        ).astype(int)
    except Exception:
        return None

    selected_parts = []
    for size_group in SIZE_GROUPS_TO_TRADE:
        sg = out[out["size_group"] == size_group].copy()
        if len(sg) < MIN_STOCKS_PER_SIZE_GROUP:
            continue

        n_select = calc_select_count(len(sg), TOP_PCT)
        sg = sg.sort_values([NEUTRAL_FACTOR_NAME, "instrument"], ascending=[False, True], kind="mergesort")
        selected_parts.append(sg.head(n_select))

    if not selected_parts:
        return None

    selected = pd.concat(selected_parts, ignore_index=True)

    if GROUP_CAPITAL_EQUAL:
        layer_count = selected["size_group"].nunique()
        selected["group_count"] = selected.groupby("size_group")["instrument"].transform("count")
        selected["target_weight"] = 1.0 / layer_count / selected["group_count"]
    else:
        selected["target_weight"] = 1.0 / len(selected)

    return selected[["date", "instrument", "size_group", NEUTRAL_FACTOR_NAME, "target_weight"]]


def build_signal_df(signal_panel, signal_to_execution):
    progress("开始逐截面中性化、15档市值分组与选股")

    selected_parts = []
    all_signal_dates = sorted(signal_panel["date"].drop_duplicates())

    for i, dt in enumerate(all_signal_dates, 1):
        g = signal_panel.loc[signal_panel["date"] == dt]
        if VERBOSE and (i == 1 or i % 5 == 0 or i == len(all_signal_dates)):
            progress(f"选股进度：{i}/{len(all_signal_dates)}，信号日 {to_date_str(dt)}，样本 {len(g):,}")

        sel = neutralize_bp_group_and_select(g)
        if sel is not None and len(sel) > 0:
            selected_parts.append(sel)

    if not selected_parts:
        raise ValueError("没有形成任何有效选股结果，请检查参数或数据。")

    selected_df = pd.concat(selected_parts, ignore_index=True)
    selected_df["signal_date"] = selected_df["date"].dt.strftime("%Y-%m-%d")
    selected_df["execution_date"] = selected_df["signal_date"].map(signal_to_execution)
    selected_df = selected_df.dropna(subset=["execution_date"]).copy()

    signal_df = selected_df[[
        "signal_date", "execution_date", "instrument", "size_group", NEUTRAL_FACTOR_NAME, "target_weight"
    ]].copy()

    signal_df = signal_df.rename(columns={"signal_date": "date"})
    signal_df["date"] = pd.to_datetime(signal_df["date"]).dt.strftime("%Y-%m-%d")
    signal_df["instrument"] = signal_df["instrument"].astype(str)

    signal_df = signal_df.sort_values(
        ["date", "size_group", NEUTRAL_FACTOR_NAME, "instrument"],
        ascending=[True, True, False, True],
        kind="mergesort"
    ).reset_index(drop=True)

    progress_rows("最终交易信号", signal_df)

    # 默认不展示中间交易信号摘要表，避免干扰正式回测输出。
    # 若需要调试选股数量和权重，可将 DISPLAY_SIGNAL_SUMMARY 改为 True。
    if DISPLAY_SIGNAL_SUMMARY:
        signal_summary = (
            signal_df.groupby("date")
            .agg(
                execution_date=("execution_date", "first"),
                stock_count=("instrument", "count"),
                avg_weight=("target_weight", "mean"),
                min_weight=("target_weight", "min"),
                max_weight=("target_weight", "max"),
            )
            .reset_index()
        )
        progress("交易信号摘要：")
        display(signal_summary.head(20))

    return signal_df


# =========================
# 6. BigTrader 原生回测
# =========================

DEFENSIVE_BANK_INSTRUMENTS = [
    to_bigtrader_instrument(x) for x in DEFENSIVE_BANK_ASSETS.keys()
]


def make_engine_inputs(signal_df, trade_dates, trend_allocation_df):
    """
    构造 BigTrader 输入。

    为了每日检查指数趋势，backtest_data 需要覆盖每个交易日。
    为控制内存，只订阅“历史上曾进入目标池的股票 + 三只防御银行股”。
    """
    signal_df = signal_df.copy()
    signal_df["instrument"] = signal_df["instrument"].map(to_bigtrader_instrument)

    signal_by_date = {
        d: g[["instrument", "target_weight"]].drop_duplicates("instrument").copy()
        for d, g in signal_df.groupby("date", sort=True)
    }

    all_backtest_instruments = sorted(
        set(signal_df["instrument"].dropna().astype(str).unique().tolist()) |
        set(DEFENSIVE_BANK_INSTRUMENTS)
    )

    backtest_dates = pd.to_datetime(trade_dates)
    backtest_dates = backtest_dates[
        (backtest_dates >= pd.to_datetime(START_DATE)) &
        (backtest_dates <= pd.to_datetime(END_DATE))
    ]
    backtest_date_strings = [to_date_str(x) for x in backtest_dates]

    # 每日触发数据：仅包含必要三列，换取趋势切换的每日响应能力。
    backtest_data = pd.MultiIndex.from_product(
        [backtest_date_strings, all_backtest_instruments],
        names=["date", "instrument"]
    ).to_frame(index=False)
    backtest_data["target_weight"] = np.float32(0.0)

    trend_for_bt = trend_allocation_df.loc[
        (trend_allocation_df.index >= pd.to_datetime(START_DATE)) &
        (trend_allocation_df.index <= pd.to_datetime(END_DATE))
    ].copy()

    stock_exposure_by_date = {
        to_date_str(d): float(row["stock_exposure"])
        for d, row in trend_for_bt.iterrows()
    }
    defensive_exposure_by_date = {
        to_date_str(d): float(row["defensive_exposure"])
        for d, row in trend_for_bt.iterrows()
    }
    risk_on_by_date = {
        to_date_str(d): bool(row["risk_on"])
        for d, row in trend_for_bt.iterrows()
    }

    progress_rows("BigTrader每日触发订阅数据", backtest_data)
    progress(
        "防御资产：" +
        "、".join([f"{name}({to_bigtrader_instrument(code)})" for code, name in DEFENSIVE_BANK_ASSETS.items()])
    )
    progress(
        f"BigTrader订阅标的数量：{len(all_backtest_instruments):,}；"
        f"每日趋势仓位日期数量：{len(stock_exposure_by_date):,}"
    )

    return (
        backtest_data,
        signal_by_date,
        stock_exposure_by_date,
        defensive_exposure_by_date,
        risk_on_by_date,
        all_backtest_instruments,
    )


def make_callbacks(
    backtest_data,
    signal_by_date,
    stock_exposure_by_date,
    defensive_exposure_by_date,
    risk_on_by_date,
    all_backtest_instruments,
):
    def initialize(context):
        try:
            context.set_commission(
                bigtrader.PerOrder(
                    buy_cost=BUY_COST,
                    sell_cost=SELL_COST,
                    min_cost=MIN_COMMISSION,
                )
            )
        except Exception as e:
            print(f"设置手续费失败，将使用引擎默认费率。原因：{e}", flush=True)

        context.signal_by_date = signal_by_date
        context.rebalance_dates = set(signal_by_date.keys())

        context.stock_exposure_by_date = stock_exposure_by_date
        context.defensive_exposure_by_date = defensive_exposure_by_date
        context.risk_on_by_date = risk_on_by_date

        context.defensive_bank_instruments = DEFENSIVE_BANK_INSTRUMENTS
        context.defensive_bank_names = {
            to_bigtrader_instrument(code): name
            for code, name in DEFENSIVE_BANK_ASSETS.items()
        }

        context.current_factor_weights = {}
        context.current_target_date = None
        context.current_stock_exposure = None
        context.current_defensive_exposure = None

        try:
            context.subscribe_bar(all_backtest_instruments, "1d", None)
        except Exception:
            pass

        print(
            f"initialize 完成：BP因子 + 中证1000 {TREND_MA_WINDOW}日均线防御性收益补偿策略",
            flush=True,
        )

    def _get_stock_exposure(context, date_str):
        try:
            return float(context.stock_exposure_by_date.get(date_str, RISK_ON_STOCK_EXPOSURE))
        except Exception:
            return float(RISK_ON_STOCK_EXPOSURE)

    def _get_defensive_exposure(context, date_str):
        try:
            return float(context.defensive_exposure_by_date.get(date_str, RISK_ON_DEFENSIVE_EXPOSURE))
        except Exception:
            return float(RISK_ON_DEFENSIVE_EXPOSURE)

    def _allocation_changed(context, stock_exposure, defensive_exposure):
        old_stock = getattr(context, "current_stock_exposure", None)
        old_def = getattr(context, "current_defensive_exposure", None)
        if old_stock is None or old_def is None:
            return True
        return (
            abs(float(old_stock) - float(stock_exposure)) > 1e-8 or
            abs(float(old_def) - float(defensive_exposure)) > 1e-8
        )

    def handle_data(context, data):
        current_date = get_current_date_from_engine(context, data)
        if current_date is None:
            return

        stock_exposure = _get_stock_exposure(context, current_date)
        defensive_exposure = _get_defensive_exposure(context, current_date)
        risk_on = bool(context.risk_on_by_date.get(current_date, True))

        is_rebalance_date = current_date in context.rebalance_dates
        allocation_changed = _allocation_changed(context, stock_exposure, defensive_exposure)

        # 非调仓日且趋势仓位没有变化时不重复下单。
        if (not is_rebalance_date) and (not allocation_changed):
            return

        # 调仓日更新因子目标池；趋势变化日沿用上一期目标池，只调整股票/银行仓位比例。
        if is_rebalance_date:
            today_signal = context.signal_by_date.get(current_date)
            if today_signal is None or len(today_signal) == 0:
                factor_weights_base = {}
            else:
                # 信号中的 target_weight 已经反映“组间等资金/整体等权”等原始因子组合权重，合计约为1。
                factor_weights_base = dict(
                    zip(
                        today_signal["instrument"].astype(str),
                        today_signal["target_weight"].astype(float),
                    )
                )
            context.current_factor_weights = factor_weights_base
            context.current_target_date = current_date
        else:
            factor_weights_base = dict(getattr(context, "current_factor_weights", {}))

        defensive_targets = list(getattr(context, "defensive_bank_instruments", []))
        factor_target_set = set(factor_weights_base.keys())
        defensive_target_set = set(defensive_targets)
        all_target_set = factor_target_set | defensive_target_set

        positions = get_positions_dict(context)
        holding_instruments = set()
        for ins, pos in positions.items():
            if position_amount(pos) > 0:
                holding_instruments.add(to_bigtrader_instrument(ins))

        # 清理不在因子目标池、也不在防御银行池中的持仓。
        for ins in sorted(holding_instruments - all_target_set):
            order_to_target_percent(context, ins, 0.0)

        # 防御银行仓位：风险关闭时为90%，三只银行等权；风险开启时为0。
        defensive_weight = 0.0
        if len(defensive_targets) > 0:
            defensive_weight = float(defensive_exposure) / len(defensive_targets)

        for ins in sorted(defensive_targets):
            order_to_target_percent(context, ins, defensive_weight)

        # 因子股票仓位：风险开启时恢复原权重；风险关闭时整体压缩到10%。
        for ins in sorted(factor_weights_base.keys()):
            base_w = float(factor_weights_base[ins])
            order_to_target_percent(context, ins, float(stock_exposure) * base_w)

        if is_rebalance_date and len(factor_weights_base) == 0:
            for ins in sorted(holding_instruments | defensive_target_set):
                order_to_target_percent(context, ins, 0.0)

        context.current_stock_exposure = stock_exposure
        context.current_defensive_exposure = defensive_exposure

        if PRINT_TREND_SWITCH_LOG:
            state_text = f"风险开启/中证1000在{TREND_MA_WINDOW}日均线上方" if risk_on else f"风险关闭/中证1000跌破{TREND_MA_WINDOW}日均线"
            if is_rebalance_date:
                print(
                    f"{current_date} 调仓：{state_text}，因子股票 {len(factor_weights_base)} 只，"
                    f"因子股票总仓位 {stock_exposure:.2%}，银行总仓位 {defensive_exposure:.2%}，"
                    f"单只银行 {defensive_weight:.4f}",
                    flush=True,
                )
            elif allocation_changed:
                print(
                    f"{current_date} 趋势仓位切换：{state_text}，沿用 {context.current_target_date} 目标池，"
                    f"因子股票总仓位 {stock_exposure:.2%}，银行总仓位 {defensive_exposure:.2%}",
                    flush=True,
                )

    return initialize, handle_data


def run_bigtrader(
    backtest_data,
    signal_by_date,
    stock_exposure_by_date,
    defensive_exposure_by_date,
    risk_on_by_date,
    all_backtest_instruments,
):
    initialize, handle_data = make_callbacks(
        backtest_data,
        signal_by_date,
        stock_exposure_by_date,
        defensive_exposure_by_date,
        risk_on_by_date,
        all_backtest_instruments,
    )

    run_kwargs = dict(
        data=backtest_data,
        start_date=START_DATE,
        end_date=END_DATE,
        initialize=initialize,
        handle_data=handle_data,
        capital_base=CAPITAL_BASE,
        benchmark=BENCHMARK,
    )

    try:
        run_kwargs["market"] = bigtrader.Market.CN_STOCK
    except Exception:
        pass

    try:
        run_kwargs["frequency"] = bigtrader.Frequency.DAILY
    except Exception:
        run_kwargs["frequency"] = "1d"

    return bigtrader.run(**run_kwargs)


# =========================
# 7. 主流程
# =========================

def main():
    progress("开始执行 BP 分层策略：防御性 + 收益补偿版本")

    trade_dates, signal_dates, signal_to_execution = get_signal_dates()

    progress(f"开始构造中证1000 {TREND_MA_WINDOW}日均线防御仓位序列")
    trend_allocation_df = build_trend_allocation_df(trade_dates)
    trend_summary = trend_allocation_df.loc[
        (trend_allocation_df.index >= pd.to_datetime(START_DATE)) &
        (trend_allocation_df.index <= pd.to_datetime(END_DATE))
    ].copy()
    risk_on_days = int(trend_summary["risk_on"].sum())
    risk_off_days = int((~trend_summary["risk_on"]).sum())
    progress(
        f"趋势仓位序列完成：风险开启 {risk_on_days:,} 天，风险关闭 {risk_off_days:,} 天；"
        f"风险关闭时因子股票仓位 {RISK_OFF_STOCK_EXPOSURE:.2%}，"
        f"银行补偿仓位 {RISK_OFF_DEFENSIVE_EXPOSURE:.2%}"
    )

    signal_panel = query_signal_panel(signal_dates)
    signal_df = build_signal_df(signal_panel, signal_to_execution)

    # 释放大表，减少 BigTrader 阶段内存压力
    del signal_panel
    gc.collect()

    (
        backtest_data,
        signal_by_date,
        stock_exposure_by_date,
        defensive_exposure_by_date,
        risk_on_by_date,
        all_backtest_instruments,
    ) = make_engine_inputs(signal_df, trade_dates, trend_allocation_df)

    del trend_allocation_df
    gc.collect()

    progress("开始运行 BigTrader 原生回测")
    performance = run_bigtrader(
        backtest_data,
        signal_by_date,
        stock_exposure_by_date,
        defensive_exposure_by_date,
        risk_on_by_date,
        all_backtest_instruments,
    )
    progress("BigTrader 回测完成")

    try:
        display(performance.summary)
    except Exception:
        display(performance)

    return performance, signal_df, backtest_data


performance, signal_df, backtest_data = main()


从防御性 + 收益补偿策略的回测结果来看，该因子搭配这个组合效果反而变差了，核心原因在于 BP 价值因子的收益来源和中证1000趋势过滤的触发逻辑并不完全匹配。波动率、动量类因子往往对市场趋势和风险偏好变化更加敏感，当指数跌破均线时降低进攻仓位，通常可以规避一部分趋势性下跌；但 BP 因子本身是一个偏慢变量、偏价值修复逻辑的因子，它的收益往往不是在市场强趋势中持续释放，而是可能集中出现在市场风格切换、低估值资产修复、银行地产周期价值股重估等阶段。此时如果简单用中证1000跌破60日均线作为降仓信号，很可能会在价值因子即将修复或正在修复的阶段提前把仓位降到10%，导致错过 BP 因子真正贡献收益的窗口。同时，释放出来的90%仓位买入工商银行、交通银行、中国银行，本质上是把组合切换成大盘银行股防御仓位，但你的 BP 策略本身已经偏向低估值、价值、防御属性，尤其在市值行业中性化后，因子收益更多来自行业内相对低估，而不是单纯依赖市场 beta。因此，这个“防御补偿仓位”并没有像在动量、波动率因子中那样起到互补作用，反而可能形成风格重复或收益替代不足。回测结果也体现了这一点：虽然累计收益率仍然较高，但胜率下降到48.4%，最大回撤扩大到23.53%，波动率升至23.49%，说明防御切换并没有有效降低风险，反而增加了择时错误和仓位切换带来的不稳定性。换句话说，这个策略对于高进攻性、强趋势依赖的因子可能是有效的风险控制工具，但对于 BP 这种低频、价值修复型因子，它可能会破坏原本需要长期持有等待修复的收益结构，因此最终表现为拖累收益和胜率。

## 质量过滤+动量排雷策略

In [ ]:
# -*- coding: utf-8 -*-
"""
BP 因子：市值行业中性化 + 先过滤后选股 策略回测（文档核对修正版）
BigQuant BigTrader 原生回测引擎版。

策略逻辑
--------
1. 全市场股票池：
   - 剔除可识别 ST、*ST、退市股票；
   - 剔除信号日停牌/无有效行情股票；
   - 若 cn_stock_status 可用，则进一步剔除信号日状态表识别出的 ST、退市、停牌股票。
2. 计算 BP = 1 / PB。
3. 每个信号截面对 BP 做：
   - 市值 + 行业中性化；
   - 中位数 MAD 去极值；
   - Z-score 标准化。
4. 全市场按总市值从小到大分为 15 组：
   - 第 1 组 = 最小市值；
   - 第 15 组 = 最大市值。
5. 在指定市值组内，先做质量与动量排雷：
   - 剔除 ROE 位于行业后 30% 的股票；
   - 剔除过去 60 日收益位于所在市值组后 30% 的股票；
   - 剔除经营现金流为负的股票，若 OCF 字段可用。
6. 在剔除后的股票池中，按 BP 中性化标准分从高到低排序。
7. 买入剔除后股票池中 BP 因子值最大的前 15%。
6. 调仓周期：60 个交易日。
7. 交易约束：
   - 信号日只使用当日及以前可得数据；
   - 不读取下一交易日涨跌停、成交量、停牌等信息；
   - 涨跌停、停牌和无法成交由 BigTrader 原生撮合处理；
   - 考虑交易成本。
8. 文档核对修正：OCF 优先使用 cn_stock_factors.net_cffoa；60日动量优先使用 cn_stock_prefactors.momentum_60 / m_lag；
9. OCF容错：若当前 BigQuant 环境无法读取经营现金流字段，可自动跳过 OCF 过滤，避免策略中断；
9. 性能优化：
   - 只读取调仓信号日截面；
   - 过去 60 日收益使用 SQL 窗口函数只返回信号日结果；
   - 数值列降精度；
   - 行业列 category；
   - 不主动保存结果文件。
"""

import warnings
warnings.filterwarnings("ignore")

import gc
import time
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

import dai
from bigquant import bigtrader


# =========================
# 1. 参数设置
# =========================

START_DATE = "2022-01-01"
END_DATE = "2026-06-30"

FACTOR_NAME = "bp_raw"
NEUTRAL_FACTOR_NAME = "bp_mkt_ind_neutral"

# 1=最小市值组，15=最大市值组
SIZE_GROUPS_TO_TRADE = [15]
N_SIZE_GROUPS = 15

# 过滤后最终买入比例：在“质量 + 动量 + 现金流过滤后的股票池”中，
# 选择 BP 中性化标准分最大的前 15%。
FINAL_TOP_PCT = 0.05

# 当前版本按 BP 因子值排序选股，不再先选前30%。
FINAL_SCORE_MODE = "bp"

# 保留综合得分参数供后续实验使用；当前默认不启用。
COMPOSITE_WEIGHT_BP = 0.60
COMPOSITE_WEIGHT_ROE = 0.20
COMPOSITE_WEIGHT_MOM = 0.20

REBALANCE_DAYS = 60
MOM_LOOKBACK_DAYS = 60

SELECT_COUNT_METHOD = "floor"      # "floor" or "ceil"
GROUP_CAPITAL_EQUAL = True         # True：市值组等资金；False：全部入选股票整体等权

WINSOR_K = 5
MIN_OBS_PER_CROSS_SECTION = 100
MIN_STOCKS_PER_SIZE_GROUP = 20

# 过滤阈值
ROE_INDUSTRY_BOTTOM_PCT = 0.30
MOM_GROUP_BOTTOM_PCT = 0.30
REQUIRE_POSITIVE_OCF = True

# 如果当前 BigQuant 环境无法读取经营现金流字段：
# True：不中断回测，跳过“经营现金流为负”过滤，并在日志提示；
# False：严格报错。
ALLOW_SKIP_OCF_IF_UNAVAILABLE = True

# 运行时自动识别，勿手动修改。
OCF_FILTER_AVAILABLE = False

CAPITAL_BASE = 1_000_000
BENCHMARK = "000300.SH"

BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COMMISSION = 5

# 性能开关
VERBOSE = True
DISPLAY_SIGNAL_SUMMARY = False
FAST_QUERY_ONLY = False

START_DATE = pd.to_datetime(START_DATE).strftime("%Y-%m-%d")
END_DATE = pd.to_datetime(END_DATE).strftime("%Y-%m-%d")

MOM_QUERY_START_DATE = (
    pd.to_datetime(START_DATE) - pd.Timedelta(days=max(160, MOM_LOOKBACK_DAYS * 4))
).strftime("%Y-%m-%d")

if not SIZE_GROUPS_TO_TRADE:
    raise ValueError("SIZE_GROUPS_TO_TRADE 不能为空。")

bad_groups = [g for g in SIZE_GROUPS_TO_TRADE if int(g) < 1 or int(g) > N_SIZE_GROUPS]
if bad_groups:
    raise ValueError(f"SIZE_GROUPS_TO_TRADE 中存在非法市值组：{bad_groups}，有效范围为 1~{N_SIZE_GROUPS}。")

SIZE_GROUPS_TO_TRADE = sorted(set([int(x) for x in SIZE_GROUPS_TO_TRADE]))

if SELECT_COUNT_METHOD not in ["floor", "ceil"]:
    raise ValueError("SELECT_COUNT_METHOD 只能是 'floor' 或 'ceil'。")

if FINAL_SCORE_MODE not in ["bp", "composite"]:
    raise ValueError("FINAL_SCORE_MODE 只能是 'bp' 或 'composite'。")


# =========================
# 2. 工具函数
# =========================

_T0 = time.time()


def _elapsed():
    sec = int(time.time() - _T0)
    return f"{sec // 60:02d}:{sec % 60:02d}"


def progress(msg):
    if VERBOSE:
        print(f"[{_elapsed()}] {msg}", flush=True)


def progress_rows(name, df):
    if VERBOSE:
        progress(f"{name}：{len(df):,} 行")


def query_df(sql, start_date=None, end_date=None):
    """
    BigQuant DAI 对分区表要求通过 filters 指定分区范围。
    统一传入 date filters，避免复杂 JOIN 下出现分区扫描权限错误。
    """
    fs = START_DATE if start_date is None else pd.to_datetime(start_date).strftime("%Y-%m-%d")
    fe = END_DATE if end_date is None else pd.to_datetime(end_date).strftime("%Y-%m-%d")
    return dai.query(sql, filters={"date": [fs, fe]}).df()


def try_query(sql, start_date=None, end_date=None):
    try:
        df = query_df(sql, start_date=start_date, end_date=end_date)
        if df is not None and len(df) > 0:
            return df
    except Exception as e:
        if VERBOSE:
            progress(f"SQL尝试失败：{str(e)[:140]}")
    return None


def first_success_query(sql_list, err_msg, start_date=None, end_date=None):
    last_error = None
    for i, sql in enumerate(sql_list, 1):
        try:
            df = query_df(sql, start_date=start_date, end_date=end_date)
            if df is not None and len(df) > 0:
                if VERBOSE:
                    progress(f"SQL第 {i} 个方案成功。")
                return df
        except Exception as e:
            last_error = e
            if FAST_QUERY_ONLY:
                break
            continue
    raise ValueError(f"{err_msg}。最后一次错误：{last_error}")


def date_in_sql(dates):
    return ", ".join([f"'{pd.to_datetime(x).strftime('%Y-%m-%d')}'" for x in dates])


def downcast_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce", downcast="float")
    return df


def to_date_str(x):
    return pd.to_datetime(x).strftime("%Y-%m-%d")


def calc_select_count(n, pct):
    if SELECT_COUNT_METHOD == "ceil":
        return max(1, int(np.ceil(n * pct)))
    return max(1, int(np.floor(n * pct)))


def winsorize_mad_array(x, k=WINSOR_K):
    x = np.asarray(x, dtype=np.float64)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    if not np.isfinite(mad) or mad <= 0:
        return x
    return np.clip(x, med - k * mad, med + k * mad)


def zscore_array(x):
    x = np.asarray(x, dtype=np.float64)
    mu = np.nanmean(x)
    sd = np.nanstd(x, ddof=1)
    if not np.isfinite(sd) or sd <= 0:
        return np.full(len(x), np.nan)
    return (x - mu) / sd


def normalize_industry_name(x):
    if pd.isna(x):
        return "UNKNOWN"
    s = str(x).strip()
    for token in [
        "申万一级行业", "申万一级", "申万", "SW2021", "SW", "中信一级行业", "中信一级",
        "一级行业", "行业", "（", "）", "(", ")", " ", "-", "_"
    ]:
        s = s.replace(token, "")
    alias = {
        "非银金融": "非银行金融",
        "非银行业金融": "非银行金融",
        "商业贸易": "商贸零售",
        "商贸零售业": "商贸零售",
        "食品饮料业": "食品饮料",
        "农林牧渔业": "农林牧渔",
        "轻工制造业": "轻工制造",
        "交通运输业": "交通运输",
        "建筑材料": "建材",
        "银行业": "银行",
        "通信设备": "通信",
        "医药生物": "医药",
    }
    return alias.get(s, s) if s else "UNKNOWN"


def is_bad_stock_name(name):
    txt = str(name).upper() if not pd.isna(name) else ""
    return ("ST" in txt) or ("退" in txt) or ("DELIST" in txt) or ("终止" in txt) or ("摘牌" in txt)


def bool_from_col(s):
    if s.dtype == bool:
        return s.fillna(False)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float) != 0
    txt = s.astype(str).str.upper().fillna("")
    return (
        txt.isin(["1", "TRUE", "T", "YES", "Y"])
        | txt.str.contains("ST|退|DELIST|SUSPEND|停牌|暂停|终止|摘牌", regex=True, na=False)
    )


def first_existing_col(df, candidates):
    cols = list(df.columns)
    lower_map = {str(c).lower(): c for c in cols}
    for c in candidates:
        if c in df.columns:
            return c
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None


def get_current_date_from_engine(context, data):
    if data is not None and hasattr(data, "current_dt"):
        return pd.to_datetime(data.current_dt).strftime("%Y-%m-%d")
    for attr in ["current_dt", "now", "current_date"]:
        if hasattr(context, attr):
            v = getattr(context, attr)
            if v is not None:
                return pd.to_datetime(v).strftime("%Y-%m-%d")
    return None


def get_positions_dict(context):
    for method in ["get_account_positions", "get_positions"]:
        if hasattr(context, method):
            try:
                pos = getattr(context, method)()
                if pos is not None:
                    return pos
            except Exception:
                pass
    return {}


def position_amount(pos_obj):
    try:
        return float(getattr(pos_obj, "amount", 0))
    except Exception:
        try:
            return float(pos_obj.get("amount", 0))
        except Exception:
            return 0.0


def order_to_target_percent(context, instrument, weight):
    try:
        context.order_target_percent(instrument, float(weight))
        return True
    except Exception:
        try:
            context.order_percent(instrument, float(weight))
            return True
        except Exception as e:
            print(f"下单失败：{instrument}, target={weight:.6f}, err={e}", flush=True)
            return False


# =========================
# 3. 交易日和信号日
# =========================

def get_signal_dates():
    progress("开始获取交易日列表")

    sql = f"""
    SELECT DISTINCT date
    FROM cn_stock_bar1d
    WHERE date >= '{START_DATE}'
      AND date <= '{END_DATE}'
    ORDER BY date
    """
    trade_dates_df = query_df(sql)
    trade_dates_df["date"] = pd.to_datetime(trade_dates_df["date"])
    trade_dates = trade_dates_df["date"].drop_duplicates().sort_values().reset_index(drop=True)

    if len(trade_dates) < REBALANCE_DAYS + 2:
        raise ValueError("指定时间段内交易日过少，无法完成调仓回测。")

    raw_signal_dates = trade_dates.iloc[::REBALANCE_DAYS].tolist()

    signal_to_execution = {}
    for dt in raw_signal_dates:
        idx = trade_dates[trade_dates == dt].index
        if len(idx) == 0:
            continue
        next_idx = int(idx[0]) + 1
        if next_idx < len(trade_dates):
            signal_to_execution[to_date_str(dt)] = to_date_str(trade_dates.iloc[next_idx])

    signal_dates = [pd.to_datetime(k) for k in signal_to_execution.keys()]
    if len(signal_dates) == 0:
        raise ValueError("没有可用调仓信号日期。")

    progress(f"交易日数量：{len(trade_dates):,}；调仓信号截面数量：{len(signal_dates):,}")
    return signal_dates, signal_to_execution


# =========================
# 4. 数据读取
# =========================

def query_base_panel(signal_dates):
    progress("开始读取调仓截面基础数据：PB、市值、行业、证券简称、有效行情")

    signal_date_sql = date_in_sql(signal_dates)

    sqls = [
        f"""
        SELECT
            v.date,
            v.instrument,
            v.pb,
            v.total_market_cap AS mkt_cap,
            p.cs_level1 AS industry,
            b.name AS stock_name,
            b.open,
            b.close
        FROM cn_stock_valuation v
        LEFT JOIN cn_stock_prefactors p
          ON v.date = p.date AND v.instrument = p.instrument
        LEFT JOIN (
            SELECT date, instrument, name, open, close
            FROM cn_stock_bar1d
            WHERE date IN ({signal_date_sql})
        ) b
          ON v.date = b.date AND v.instrument = b.instrument
        WHERE v.date IN ({signal_date_sql})
          AND v.pb IS NOT NULL
          AND v.total_market_cap IS NOT NULL
        """,
    ]

    if not FAST_QUERY_ONLY:
        sqls.extend([
            f"""
            SELECT
                v.date,
                v.instrument,
                v.pb,
                v.total_market_cap AS mkt_cap,
                p.industry AS industry,
                b.name AS stock_name,
                b.open,
                b.close
            FROM cn_stock_valuation v
            LEFT JOIN cn_stock_prefactors p
              ON v.date = p.date AND v.instrument = p.instrument
            LEFT JOIN (
                SELECT date, instrument, name, open, close
                FROM cn_stock_bar1d
                WHERE date IN ({signal_date_sql})
            ) b
              ON v.date = b.date AND v.instrument = b.instrument
            WHERE v.date IN ({signal_date_sql})
              AND v.pb IS NOT NULL
              AND v.total_market_cap IS NOT NULL
            """,
            f"""
            SELECT
                v.date,
                v.instrument,
                v.pb,
                v.total_mv AS mkt_cap,
                p.cs_level1 AS industry,
                b.name AS stock_name,
                b.open,
                b.close
            FROM cn_stock_valuation v
            LEFT JOIN cn_stock_prefactors p
              ON v.date = p.date AND v.instrument = p.instrument
            LEFT JOIN (
                SELECT date, instrument, name, open, close
                FROM cn_stock_bar1d
                WHERE date IN ({signal_date_sql})
            ) b
              ON v.date = b.date AND v.instrument = b.instrument
            WHERE v.date IN ({signal_date_sql})
              AND v.pb IS NOT NULL
              AND v.total_mv IS NOT NULL
            """,
            f"""
            SELECT
                v.date,
                v.instrument,
                v.pb,
                v.total_market_cap AS mkt_cap,
                'UNKNOWN' AS industry,
                b.name AS stock_name,
                b.open,
                b.close
            FROM cn_stock_valuation v
            LEFT JOIN (
                SELECT date, instrument, name, open, close
                FROM cn_stock_bar1d
                WHERE date IN ({signal_date_sql})
            ) b
              ON v.date = b.date AND v.instrument = b.instrument
            WHERE v.date IN ({signal_date_sql})
              AND v.pb IS NOT NULL
              AND v.total_market_cap IS NOT NULL
            """,
        ])

    df = first_success_query(sqls, "未能读取调仓截面基础数据")
    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["industry"] = df["industry"].fillna("UNKNOWN").map(normalize_industry_name).astype("category")
    df["stock_name"] = df["stock_name"].fillna("").astype(str)

    downcast_numeric(df, ["pb", "mkt_cap", "open", "close"])

    df = df.dropna(subset=["date", "instrument", "pb", "mkt_cap"])
    df = df[(df["pb"] > 0) & (df["mkt_cap"] > 0)].copy()

    # 信号日停牌/无有效行情过滤：用信号日 open/close 是否有效做轻量近似。
    before = len(df)
    df = df[df["open"].notna() & df["close"].notna() & (df["open"] > 0) & (df["close"] > 0)].copy()
    progress(f"信号日无有效行情/疑似停牌过滤：剔除 {before - len(df):,} 行；剩余 {len(df):,} 行")

    # 信号日名称过滤 ST、退市。
    before = len(df)
    bad_name = df["stock_name"].map(is_bad_stock_name).astype(bool)
    df = df[~bad_name].copy()
    progress(f"证券简称过滤 ST/退市：剔除 {before - len(df):,} 行；剩余 {len(df):,} 行")

    df[FACTOR_NAME] = 1.0 / df["pb"].astype(float)
    df = df[np.isfinite(df[FACTOR_NAME])].copy()
    downcast_numeric(df, [FACTOR_NAME])

    df = df.sort_values(["date", "instrument"], kind="mergesort")
    df = df.drop_duplicates(subset=["date", "instrument"], keep="last")

    df = df[["date", "instrument", FACTOR_NAME, "mkt_cap", "industry"]].reset_index(drop=True)
    progress_rows("基础信号截面数据", df)
    return df


def query_status_panel(signal_dates):
    """
    轻量读取 cn_stock_status：若不可用则跳过。
    只使用信号日状态，不使用未来状态。
    """
    progress("尝试读取信号日 ST/退市/停牌状态表")

    signal_date_sql = date_in_sql(signal_dates)
    sql = f"""
    SELECT *
    FROM cn_stock_status
    WHERE date IN ({signal_date_sql})
    """
    status = try_query(sql)
    if status is None or len(status) == 0:
        progress("未读取到 cn_stock_status，有效行情与证券简称过滤仍会生效。")
        return None

    if "date" not in status.columns or "instrument" not in status.columns:
        progress("cn_stock_status 缺少 date 或 instrument，跳过状态表过滤。")
        return None

    status["date"] = pd.to_datetime(status["date"])
    status["instrument"] = status["instrument"].astype(str)

    st_col = first_existing_col(status, ["is_st", "st", "st_status", "risk_warning", "special_treatment"])
    sus_col = first_existing_col(status, ["is_suspended", "suspended", "suspend", "paused", "is_paused", "trade_status"])
    delist_col = first_existing_col(status, ["is_delist", "delist_status", "delisted", "list_status", "listing_status", "status"])

    bad = pd.Series(False, index=status.index)

    if st_col:
        bad = bad | bool_from_col(status[st_col])
    if sus_col:
        bad = bad | bool_from_col(status[sus_col])
    if delist_col:
        bad = bad | bool_from_col(status[delist_col])

    # 若状态表中存在名称字段，也一起识别。
    name_col = first_existing_col(status, ["name", "stock_name", "short_name", "sec_name", "security_name"])
    if name_col:
        bad = bad | status[name_col].map(is_bad_stock_name).astype(bool)

    out = status[["date", "instrument"]].copy()
    out["is_bad_status"] = bad.astype(bool)
    out = out.sort_values(["date", "instrument", "is_bad_status"], kind="mergesort")
    out = out.drop_duplicates(subset=["date", "instrument"], keep="last")
    progress_rows("状态表过滤数据", out)
    return out


def query_quality_metric(signal_dates, metric_name, field_candidates, table_candidates, required=True):
    """
    尝试从多个表/字段读取单个质量指标。
    返回 date, instrument, metric_name。

    对 OCF 这类平台字段差异较大的指标，可设置 required=False；
    读取失败时返回 None，不中断主流程。
    """
    signal_date_sql = date_in_sql(signal_dates)

    sqls = []
    for table in table_candidates:
        for field in field_candidates:
            sqls.append(
                f"""
                SELECT
                    date,
                    instrument,
                    {field} AS {metric_name}
                FROM {table}
                WHERE date IN ({signal_date_sql})
                  AND {field} IS NOT NULL
                """
            )

    try:
        df = first_success_query(sqls, f"未能读取质量指标 {metric_name}")
    except Exception as e:
        if required:
            raise
        progress(f"警告：未能读取质量指标 {metric_name}，将跳过对应过滤。错误摘要：{str(e)[:220]}")
        return None

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df[metric_name] = pd.to_numeric(df[metric_name], errors="coerce")
    df = df.dropna(subset=["date", "instrument", metric_name])
    df = df[np.isfinite(df[metric_name])].copy()
    df = df.sort_values(["date", "instrument"], kind="mergesort")
    df = df.drop_duplicates(subset=["date", "instrument"], keep="last")
    downcast_numeric(df, [metric_name])
    progress_rows(f"质量指标 {metric_name}", df)
    return df[["date", "instrument", metric_name]]


def query_roe_panel(signal_dates):
    progress("开始读取 ROE 指标")
    table_candidates = [
        "cn_stock_prefactors",
        "cn_stock_factors",
        "cn_stock_financial_indicator",
        "cn_stock_fina_indicator",
        "cn_stock_derivative_indicator",
    ]
    field_candidates = [
        "roe_ttm",
        "roe",
        "roe_avg",
        "return_on_equity",
        "roe_weighted",
        "net_asset_income_rate",
    ]
    return query_quality_metric(signal_dates, "roe", field_candidates, table_candidates)


def query_ocf_panel(signal_dates):
    """
    读取经营现金流指标。

    文档核对后的优先口径：
    - BigQuant 质量投资策略示例中，经营活动现金流字段使用 cn_stock_factors.net_cffoa；
    - 因此这里优先读取 cn_stock_factors.net_cffoa 作为 OCF 符号判断字段；
    - 若字段不可用，再尝试少量官方相关财务因子表，避免长时间遍历大量不存在的表。
    """
    progress("开始读取经营现金流指标")

    signal_date_sql = date_in_sql(signal_dates)

    sqls = [
        f"""
        SELECT
            date,
            instrument,
            net_cffoa AS ocf
        FROM cn_stock_factors
        WHERE date IN ({signal_date_sql})
          AND net_cffoa IS NOT NULL
        """,
        f"""
        SELECT
            date,
            instrument,
            net_cffoa AS ocf
        FROM cn_stock_prefactors
        WHERE date IN ({signal_date_sql})
          AND net_cffoa IS NOT NULL
        """,
        f"""
        SELECT
            date,
            instrument,
            net_cffoa_lf AS ocf
        FROM cn_stock_factors_financial_items
        WHERE date IN ({signal_date_sql})
          AND net_cffoa_lf IS NOT NULL
        """,
        f"""
        SELECT
            date,
            instrument,
            net_cffoa_ttm AS ocf
        FROM cn_stock_factors_financial_items
        WHERE date IN ({signal_date_sql})
          AND net_cffoa_ttm IS NOT NULL
        """,
        f"""
        SELECT
            date,
            instrument,
            cfoa AS ocf
        FROM cn_stock_factors_financial_indicators
        WHERE date IN ({signal_date_sql})
          AND cfoa IS NOT NULL
        """,
        f"""
        SELECT
            date,
            instrument,
            cfoa AS ocf
        FROM cn_stock_facrtors_financial_extend
        WHERE date IN ({signal_date_sql})
          AND cfoa IS NOT NULL
        """,
    ]

    try:
        df = first_success_query(
            sqls,
            "未能读取经营现金流字段 ocf",
            start_date=START_DATE,
            end_date=END_DATE,
        )
    except Exception as e:
        if not ALLOW_SKIP_OCF_IF_UNAVAILABLE:
            raise
        progress(f"警告：未能读取经营现金流字段，将跳过 OCF 过滤。错误摘要：{str(e)[:220]}")
        return None

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["ocf"] = pd.to_numeric(df["ocf"], errors="coerce")
    df = df.dropna(subset=["date", "instrument", "ocf"])
    df = df[np.isfinite(df["ocf"])].copy()
    df = df.sort_values(["date", "instrument"], kind="mergesort")
    df = df.drop_duplicates(subset=["date", "instrument"], keep="last")
    downcast_numeric(df, ["ocf"])
    progress_rows("经营现金流指标 ocf", df)
    return df[["date", "instrument", "ocf"]]


def query_momentum_panel(signal_dates):
    """
    计算过去 N 日收益率。

    文档核对后的实现：
    - BigQuant cn_stock_prefactors 是 view，带时间窗口因子需要向前多读取一段数据，
      再在 pandas 中过滤到正式回测区间；
    - 优先使用 cn_stock_prefactors.momentum_60；
    - 如果字段不存在，使用官方示例同类写法 close / m_lag(close, N) - 1；
    - 最后才回退到 Python groupby shift。
    """
    progress(f"开始计算过去 {MOM_LOOKBACK_DAYS} 日收益率")

    mom_col = f"ret_{MOM_LOOKBACK_DAYS}d"
    signal_date_set = set(pd.to_datetime(signal_dates).strftime("%Y-%m-%d"))

    # 方案1：直接读取预计算 momentum_60。按官方说明，向前多取数据后再过滤正式日期。
    prefactor_field = f"momentum_{MOM_LOOKBACK_DAYS}"
    sql_prefactor = f"""
    SELECT
        date,
        instrument,
        {prefactor_field} AS {mom_col}
    FROM cn_stock_prefactors
    WHERE date >= '{MOM_QUERY_START_DATE}'
      AND date <= '{END_DATE}'
      AND {prefactor_field} IS NOT NULL
    """
    mom = try_query(sql_prefactor, start_date=MOM_QUERY_START_DATE, end_date=END_DATE)

    # 方案2：使用 DAI 时间序列算子 m_lag，避免标准 lag/window 在部分环境里不返回结果。
    if mom is None or len(mom) == 0:
        progress(f"未能直接读取 cn_stock_prefactors.{prefactor_field}，尝试用 m_lag 计算。")
        sql_mlag = f"""
        SELECT
            date,
            instrument,
            close / m_lag(close, {MOM_LOOKBACK_DAYS}) - 1 AS {mom_col}
        FROM cn_stock_bar1d
        WHERE date >= '{MOM_QUERY_START_DATE}'
          AND date <= '{END_DATE}'
          AND close > 0
        """
        mom = try_query(sql_mlag, start_date=MOM_QUERY_START_DATE, end_date=END_DATE)

    # 方案3：Python fallback。只有前两种失败才触发，避免常规情况下占用较大内存。
    if mom is None or len(mom) == 0:
        progress("m_lag 计算失败，回退到 Python groupby shift 计算60日收益。")
        sql_bar = f"""
        SELECT
            date,
            instrument,
            close
        FROM cn_stock_bar1d
        WHERE date >= '{MOM_QUERY_START_DATE}'
          AND date <= '{END_DATE}'
          AND close > 0
        ORDER BY instrument, date
        """
        bar = first_success_query(
            [sql_bar],
            f"未能读取 cn_stock_bar1d 计算过去 {MOM_LOOKBACK_DAYS} 日收益率",
            start_date=MOM_QUERY_START_DATE,
            end_date=END_DATE,
        )
        bar["date"] = pd.to_datetime(bar["date"])
        bar["instrument"] = bar["instrument"].astype(str)
        bar["close"] = pd.to_numeric(bar["close"], errors="coerce", downcast="float")
        bar = bar.dropna(subset=["date", "instrument", "close"])
        bar = bar[bar["close"] > 0].copy()
        bar = bar.sort_values(["instrument", "date"], kind="mergesort")
        bar[mom_col] = bar.groupby("instrument", sort=False)["close"].pct_change(MOM_LOOKBACK_DAYS)
        mom = bar[["date", "instrument", mom_col]].copy()
        del bar
        gc.collect()

    mom["date"] = pd.to_datetime(mom["date"])
    mom["date_str"] = mom["date"].dt.strftime("%Y-%m-%d")
    mom["instrument"] = mom["instrument"].astype(str)
    mom[mom_col] = pd.to_numeric(mom[mom_col], errors="coerce")
    mom = mom[mom["date_str"].isin(signal_date_set)].copy()
    mom = mom.dropna(subset=["date", "instrument", mom_col])
    mom = mom[np.isfinite(mom[mom_col])].copy()
    mom = mom.sort_values(["date", "instrument"], kind="mergesort")
    mom = mom.drop_duplicates(subset=["date", "instrument"], keep="last")
    downcast_numeric(mom, [mom_col])
    progress_rows(f"{MOM_LOOKBACK_DAYS}日动量数据", mom)
    return mom[["date", "instrument", mom_col]]


def build_signal_panel(signal_dates):
    base = query_base_panel(signal_dates)

    status = query_status_panel(signal_dates)
    if status is not None:
        before = len(base)
        base = base.merge(status, on=["date", "instrument"], how="left")
        base["is_bad_status"] = base["is_bad_status"].fillna(False).astype(bool)
        bad_n = int(base["is_bad_status"].sum())
        base = base[~base["is_bad_status"]].drop(columns=["is_bad_status"]).copy()
        progress(f"状态表过滤 ST/退市/停牌：剔除 {bad_n:,} 行；剩余 {len(base):,} 行；过滤前 {before:,} 行")

    global OCF_FILTER_AVAILABLE

    roe = query_roe_panel(signal_dates)
    ocf = query_ocf_panel(signal_dates)
    mom = query_momentum_panel(signal_dates)

    panel = base.merge(roe, on=["date", "instrument"], how="inner")
    del roe
    gc.collect()

    if ocf is not None and len(ocf) > 0:
        OCF_FILTER_AVAILABLE = True
        panel = panel.merge(ocf, on=["date", "instrument"], how="inner")
        del ocf
        gc.collect()
    else:
        OCF_FILTER_AVAILABLE = False
        panel["ocf"] = np.nan
        progress("警告：经营现金流字段不可用，本次回测跳过“经营现金流为负”过滤。")

    panel = panel.merge(mom, on=["date", "instrument"], how="inner")
    del mom, base
    gc.collect()

    required_cols = [FACTOR_NAME, "mkt_cap", "industry", "roe", f"ret_{MOM_LOOKBACK_DAYS}d"]
    panel = panel.dropna(subset=required_cols)
    finite_mask = (
        (panel["mkt_cap"] > 0)
        & np.isfinite(panel[FACTOR_NAME])
        & np.isfinite(panel["roe"])
        & np.isfinite(panel[f"ret_{MOM_LOOKBACK_DAYS}d"])
    )

    if OCF_FILTER_AVAILABLE:
        finite_mask = finite_mask & np.isfinite(panel["ocf"])

    panel = panel[finite_mask].copy()

    if REQUIRE_POSITIVE_OCF and OCF_FILTER_AVAILABLE:
        before = len(panel)
        panel = panel[panel["ocf"] >= 0].copy()
        progress(f"经营现金流为负过滤：剔除 {before - len(panel):,} 行；剩余 {len(panel):,} 行")
    elif REQUIRE_POSITIVE_OCF and not OCF_FILTER_AVAILABLE:
        progress("经营现金流过滤未执行：当前环境未能读取经营现金流字段。")

    panel["industry"] = panel["industry"].astype("category")
    progress_rows("合并质量与动量后的信号截面数据", panel)
    return panel.reset_index(drop=True)


# =========================
# 5. 截面中性化、过滤、分层、选股
# =========================

def neutralize_bp_group_and_select(g):
    mom_col = f"ret_{MOM_LOOKBACK_DAYS}d"

    g = g[["date", "instrument", FACTOR_NAME, "mkt_cap", "industry", "roe", "ocf", mom_col]].copy()
    g = g.sort_values(["instrument"], kind="mergesort").reset_index(drop=True)

    if len(g) < max(MIN_OBS_PER_CROSS_SECTION, N_SIZE_GROUPS * MIN_STOCKS_PER_SIZE_GROUP // 2):
        return None

    y0 = pd.to_numeric(g[FACTOR_NAME], errors="coerce")
    cap = pd.to_numeric(g["mkt_cap"], errors="coerce")
    valid0 = y0.notna() & cap.notna() & (cap > 0)
    if valid0.sum() < MIN_OBS_PER_CROSS_SECTION:
        return None

    g = g.loc[valid0].copy()
    g["log_mkt_z"] = zscore_array(np.log(g["mkt_cap"].astype(float).values))
    g = g.dropna(subset=[FACTOR_NAME, "log_mkt_z", "roe", mom_col])
    if len(g) < MIN_OBS_PER_CROSS_SECTION:
        return None

    # FWL：行业去均值后，对市值残差做一元回归。
    ind_group = g.groupby("industry", sort=True, observed=True)
    y_dm = g[FACTOR_NAME].astype(float) - ind_group[FACTOR_NAME].transform("mean").astype(float)
    x_dm = g["log_mkt_z"].astype(float) - ind_group["log_mkt_z"].transform("mean").astype(float)

    y = y_dm.to_numpy(dtype=np.float64)
    x = x_dm.to_numpy(dtype=np.float64)
    valid = np.isfinite(y) & np.isfinite(x)

    if valid.sum() < MIN_OBS_PER_CROSS_SECTION:
        return None

    x_valid = x[valid]
    y_valid = y[valid]
    x_var = float(np.sum(x_valid * x_valid))
    if x_var <= 0 or not np.isfinite(x_var):
        return None

    beta = float(np.sum(x_valid * y_valid) / x_var)
    resid = y_valid - beta * x_valid

    out = g.loc[valid, ["date", "instrument", "mkt_cap", "industry", "roe", "ocf", mom_col]].copy()
    resid_w = winsorize_mad_array(resid, WINSOR_K)
    out[NEUTRAL_FACTOR_NAME] = zscore_array(resid_w)
    out = out.dropna(subset=[NEUTRAL_FACTOR_NAME, "mkt_cap", "roe", mom_col])

    if len(out) < MIN_OBS_PER_CROSS_SECTION:
        return None

    # 市值15组，排序固定，保证复现稳定。
    out = out.sort_values(["mkt_cap", "instrument"], ascending=[True, True], kind="mergesort").reset_index(drop=True)
    rank = out["mkt_cap"].rank(method="first", ascending=True)

    try:
        out["size_group"] = pd.qcut(
            rank,
            q=N_SIZE_GROUPS,
            labels=list(range(1, N_SIZE_GROUPS + 1))
        ).astype(int)
    except Exception:
        return None

    # ROE 行业内分位：剔除行业后30%
    out["roe_ind_pct"] = (
        out.groupby("industry", observed=True)["roe"]
        .rank(method="first", ascending=True, pct=True)
    )

    # 60日动量所在市值组分位：剔除组内后30%
    out["mom_group_pct"] = (
        out.groupby("size_group", observed=True)[mom_col]
        .rank(method="first", ascending=True, pct=True)
    )

    # 综合得分备用
    out["roe_z"] = out.groupby("industry", observed=True)["roe"].transform(lambda s: pd.Series(zscore_array(s.values), index=s.index))
    out["mom_z"] = out.groupby("size_group", observed=True)[mom_col].transform(lambda s: pd.Series(zscore_array(s.values), index=s.index))
    out["score_composite"] = (
        COMPOSITE_WEIGHT_BP * out[NEUTRAL_FACTOR_NAME].astype(float)
        + COMPOSITE_WEIGHT_ROE * out["roe_z"].fillna(0).astype(float)
        + COMPOSITE_WEIGHT_MOM * out["mom_z"].fillna(0).astype(float)
    )

    # 当前版本最终选股固定按 BP 中性化标准分排序；综合得分仅保留供后续扩展。
    score_col = NEUTRAL_FACTOR_NAME

    selected_parts = []
    for size_group in SIZE_GROUPS_TO_TRADE:
        sg_all = out[out["size_group"] == size_group].copy()
        if len(sg_all) < MIN_STOCKS_PER_SIZE_GROUP:
            continue

        # 第一步：先在指定市值组内做质量与动量排雷过滤。
        # 注意：这里不再预先选 BP 最高前30%，而是在完整市值组内先剔除价值陷阱。
        before_filter = len(sg_all)
        sg_filtered = sg_all[
            (sg_all["roe_ind_pct"] > ROE_INDUSTRY_BOTTOM_PCT)
            & (sg_all["mom_group_pct"] > MOM_GROUP_BOTTOM_PCT)
        ].copy()

        if REQUIRE_POSITIVE_OCF and OCF_FILTER_AVAILABLE:
            sg_filtered = sg_filtered[sg_filtered["ocf"] >= 0].copy()

        if len(sg_filtered) < MIN_STOCKS_PER_SIZE_GROUP:
            continue

        # 第二步：在剔除后的股票池中，按 BP 因子值从高到低排序，买入前15%。
        n_final = calc_select_count(len(sg_filtered), FINAL_TOP_PCT)
        sg_final = (
            sg_filtered
            .sort_values([NEUTRAL_FACTOR_NAME, "instrument"], ascending=[False, True], kind="mergesort")
            .head(n_final)
            .copy()
        )
        sg_final["pre_filter_count"] = before_filter
        sg_final["post_filter_count"] = len(sg_filtered)
        selected_parts.append(sg_final)

    if not selected_parts:
        return None

    selected = pd.concat(selected_parts, ignore_index=True)

    if GROUP_CAPITAL_EQUAL:
        layer_count = selected["size_group"].nunique()
        selected["group_count"] = selected.groupby("size_group")["instrument"].transform("count")
        selected["target_weight"] = 1.0 / layer_count / selected["group_count"]
    else:
        selected["target_weight"] = 1.0 / len(selected)

    return selected[[
        "date",
        "instrument",
        "size_group",
        NEUTRAL_FACTOR_NAME,
        "roe",
        "ocf",
        mom_col,
        "roe_ind_pct",
        "mom_group_pct",
        "target_weight",
    ]]


def build_signal_df(signal_panel, signal_to_execution):
    progress("开始逐截面中性化、15档市值分组、先过滤后按BP选股")

    selected_parts = []
    all_signal_dates = sorted(signal_panel["date"].drop_duplicates())

    for i, dt in enumerate(all_signal_dates, 1):
        g = signal_panel.loc[signal_panel["date"] == dt]
        if VERBOSE and (i == 1 or i % 5 == 0 or i == len(all_signal_dates)):
            progress(f"选股进度：{i}/{len(all_signal_dates)}，信号日 {to_date_str(dt)}，样本 {len(g):,}")

        sel = neutralize_bp_group_and_select(g)
        if sel is not None and len(sel) > 0:
            selected_parts.append(sel)

    if not selected_parts:
        raise ValueError("没有形成任何有效选股结果，请检查参数、质量字段或过滤条件。")

    selected_df = pd.concat(selected_parts, ignore_index=True)
    selected_df["signal_date"] = selected_df["date"].dt.strftime("%Y-%m-%d")
    selected_df["execution_date"] = selected_df["signal_date"].map(signal_to_execution)
    selected_df = selected_df.dropna(subset=["execution_date"]).copy()

    signal_df = selected_df[[
        "signal_date",
        "execution_date",
        "instrument",
        "size_group",
        NEUTRAL_FACTOR_NAME,
        "roe",
        "ocf",
        f"ret_{MOM_LOOKBACK_DAYS}d",
        "target_weight",
    ]].copy()

    signal_df = signal_df.rename(columns={"signal_date": "date"})
    signal_df["date"] = pd.to_datetime(signal_df["date"]).dt.strftime("%Y-%m-%d")
    signal_df["instrument"] = signal_df["instrument"].astype(str)

    score_col = NEUTRAL_FACTOR_NAME
    signal_df = signal_df.sort_values(
        ["date", "size_group", score_col, "instrument"],
        ascending=[True, True, False, True],
        kind="mergesort"
    ).reset_index(drop=True)

    progress_rows("最终交易信号", signal_df)

    if DISPLAY_SIGNAL_SUMMARY:
        signal_summary = (
            signal_df.groupby("date")
            .agg(
                execution_date=("execution_date", "first"),
                stock_count=("instrument", "count"),
                avg_weight=("target_weight", "mean"),
                min_weight=("target_weight", "min"),
                max_weight=("target_weight", "max"),
            )
            .reset_index()
        )
        progress("交易信号摘要：")
        display(signal_summary.head(20))

    return signal_df


# =========================
# 6. BigTrader 原生回测
# =========================

def make_engine_inputs(signal_df):
    backtest_data = signal_df[["date", "instrument", "target_weight"]].copy()
    backtest_data["date"] = pd.to_datetime(backtest_data["date"]).dt.strftime("%Y-%m-%d")
    backtest_data["instrument"] = backtest_data["instrument"].astype(str)

    signal_by_date = {
        d: g[["instrument", "target_weight"]].copy()
        for d, g in signal_df.groupby("date", sort=True)
    }

    target_by_date = {
        d: set(g["instrument"].astype(str))
        for d, g in signal_df.groupby("date", sort=True)
    }
    return backtest_data, signal_by_date, target_by_date


def make_callbacks(backtest_data, signal_by_date, target_by_date):
    def initialize(context):
        try:
            context.set_commission(
                bigtrader.PerOrder(
                    buy_cost=BUY_COST,
                    sell_cost=SELL_COST,
                    min_cost=MIN_COMMISSION,
                )
            )
        except Exception as e:
            print(f"设置手续费失败，将使用引擎默认费率。原因：{e}", flush=True)

        context.signal_by_date = signal_by_date
        context.target_by_date = target_by_date
        context.rebalance_dates = set(signal_by_date.keys())

        try:
            context.subscribe_bar(list(backtest_data["instrument"].drop_duplicates()), "1d", None)
        except Exception:
            pass

    def handle_data(context, data):
        current_date = get_current_date_from_engine(context, data)
        if current_date is None:
            return

        if current_date not in context.rebalance_dates:
            return

        today_signal = context.signal_by_date.get(current_date)
        if today_signal is None or len(today_signal) == 0:
            return

        target_weights = dict(zip(today_signal["instrument"].astype(str), today_signal["target_weight"].astype(float)))
        target_instruments = set(target_weights.keys())

        positions = get_positions_dict(context)
        holding_instruments = set()
        for ins, pos in positions.items():
            if position_amount(pos) > 0:
                holding_instruments.add(str(ins))

        # 卖出不在目标池中的股票；是否能成交由 BigTrader 按实际撮合日状态处理。
        for ins in sorted(holding_instruments - target_instruments):
            order_to_target_percent(context, ins, 0.0)

        # 买入或调整目标股票到目标权重。
        for ins in sorted(target_weights.keys()):
            order_to_target_percent(context, ins, target_weights[ins])

    return initialize, handle_data


def run_bigtrader(backtest_data, signal_by_date, target_by_date):
    initialize, handle_data = make_callbacks(backtest_data, signal_by_date, target_by_date)

    run_kwargs = dict(
        data=backtest_data,
        start_date=START_DATE,
        end_date=END_DATE,
        initialize=initialize,
        handle_data=handle_data,
        capital_base=CAPITAL_BASE,
        benchmark=BENCHMARK,
    )

    try:
        run_kwargs["market"] = bigtrader.Market.CN_STOCK
    except Exception:
        pass

    try:
        run_kwargs["frequency"] = bigtrader.Frequency.DAILY
    except Exception:
        run_kwargs["frequency"] = "1d"

    return bigtrader.run(**run_kwargs)


# =========================
# 7. 主流程
# =========================

def main():
    progress("开始执行 BP + 先过滤后选股 分层策略")

    signal_dates, signal_to_execution = get_signal_dates()
    signal_panel = build_signal_panel(signal_dates)
    signal_df = build_signal_df(signal_panel, signal_to_execution)

    del signal_panel
    gc.collect()

    backtest_data, signal_by_date, target_by_date = make_engine_inputs(signal_df)

    progress("开始运行 BigTrader 原生回测")
    performance = run_bigtrader(backtest_data, signal_by_date, target_by_date)
    progress("BigTrader 回测完成")

    try:
        display(performance.summary)
    except Exception:
        display(performance)

    return performance, signal_df, backtest_data


performance, signal_df, backtest_data = main()


结果发现，这样的策略仍然会拖累因子的表现，说明 BP 因子的收益来源并没有被“ROE 过滤、60 日动量排雷、经营现金流过滤”有效增强，反而被这些条件削弱了原本的价值暴露。从回测结果看，策略最终累计收益率为 73.52%、年化收益率 13.66%，虽然仍然显著跑赢基准，但最大回撤扩大到 39.59%，收益波动率也升至 28.24%，夏普比率只有 0.49，说明过滤后并没有改善风险收益比，反而让组合变得更集中、更波动。其根本原因在于，BP 因子本身是一个逆向价值因子，高 BP 股票往往对应市场暂时不喜欢、盈利处于低点、现金流承压或短期趋势较弱的公司，而这些股票恰恰可能是未来估值修复的主要来源。如果在选股前剔除 ROE 行业后 30%、60 日动量后 30%、经营现金流为负的股票，表面上是在过滤“价值陷阱”，但实际上也可能把一批处于周期底部、短期基本面较差、但估值修复弹性最大的股票提前剔除了。尤其是 BP 因子经过市值行业中性化后，本来剩下的已经是“行业内相对便宜”的信息，再叠加质量和动量过滤，容易导致因子暴露被过度稀释：买到的股票不再是最有价值修复弹性的低估值股票，而是变成“相对便宜但质量尚可、趋势不太差”的折中组合。这个折中组合未必有更强 Alpha，反而可能失去 BP 因子的核心收益来源。因此，这次结果说明，对 BP 这类深度价值因子，简单加入质量过滤和动量过滤并不一定能提升效果，关键在于这些过滤条件可能与价值因子的反转逻辑存在冲突：它们减少了一部分风险，但也同时削掉了低估值修复中最重要的收益弹性，最终表现为收益被拖累、回撤没有明显改善、风险调整收益下降。

## BP因子总结

从裸用版本的回测结果来看，这个 BP 因子的表现明显优于加入防御、质量过滤和动量排雷后的版本，说明它的核心收益来源并不是风险规避或优质股筛选，而是更纯粹的低估值修复和价值反转逻辑：当市场中存在大量被低估、被忽视或被阶段性错杀的股票时，BP 因子能够较好地捕捉这些股票后续估值回归带来的收益。该策略累计收益率达到 131.59%，年化收益率 21.54%，显著跑赢基准，同时胜率 72.56%、盈亏比 2.53、信息比率 0.86，说明它并不是偶然依赖少数极端行情，而是在较多调仓周期中都能贡献有效超额。但同时，策略波动率达到 29.78%，最大回撤达到 38.41%，也说明这个因子并不适合追求低波动、稳健净值的防御型场景，而更适合用于估值分化较大、市场定价效率较低、低估值股票存在修复机会的环境，尤其是在价值风格占优、成长股估值收缩、市场风险偏好从追逐高景气转向寻找安全边际时，它更容易发挥作用。此前加入中证1000趋势防御、ROE过滤、60日动量过滤、经营现金流过滤后反而拖累表现，也说明该因子的有效性可能恰恰来自一些短期质量不佳、趋势偏弱、但估值已经充分反映悲观预期的股票，如果过早剔除这些标的，就会削弱 BP 因子最重要的反转弹性。因此，这个因子更适合作为进攻型价值反转因子使用，而不是被改造成低波动质量价值因子；它适合在价值修复、低估值扩散、中小市值估值回归、市场风格从成长切换到价值的阶段发挥作用，但不适合在成长风格单边占优、资金持续追逐高估值高景气资产、或市场极端下跌导致低估值继续被压制的阶段单独重仓使用。